我重新核对了两篇论文的正文与推导结构。下面将 **DPM-Solver（2022）作为主线**，按原论文的章节组织完整笔记，再单独讲解 **DPM-Solver++**。这样可以避免把原论文的一阶、二阶、三阶算法与后续的 2S、2M 混为一谈。

还需要先纠正前一版中的两个容易误解的地方：

1. **DPM-Solver 不要求一定采用 VP 调度。** 其基本假设是高斯边缘分布 $x_t=\alpha_t x_0+\sigma_t\epsilon$，以及严格单调的信噪比；VP 是其中一种情况。
2. **DPM-Solver++ 的二阶公式有不同变体。** 精确保留指数积分矩得到的二阶更新，与某些实现中使用的简化二阶系数，不能不加区分地写成同一个公式。

以下统一约定：正向扩散时间 $t$ 从 $0$ 增大到 $T$，生成时从 $T$ 反向积分到 $0$；而半对数信噪比 $\lambda$ 在生成过程中递增。

# 第一部分：DPM-Solver 原论文

## 1. 研究问题与主要贡献

论文名称：

**DPM-Solver: A Fast ODE Solver for Diffusion Probabilistic Model Sampling in Around 10 Steps**

作者：Cheng Lu、Yuhao Zhou、Fan Bao、Jianfei Chen、Chongxuan Li、Jun Zhu。

论文的研究目标是：**不重新训练扩散模型，尽可能减少生成过程中神经网络的调用次数。**

在原论文发表时，传统扩散采样通常需要数百至数千次网络调用。通用 ODE 求解器虽然能够加速，但在约 10 次 NFE 的少步数场景下仍然面临较大数值误差。

作者发现，扩散模型的 PF ODE 具有半线性结构：

$$
\boxed{
\frac{dx_t}{dt}
=
f(t)x_t+
b(t)\epsilon_\theta(x_t,t)
}
\tag{1.1}
$$

其中：

- $f(t)x_t$ 是已知线性项；
- $b(t)$ 是由噪声调度决定的已知函数；
- $\epsilon_\theta$ 是神经网络预测项。

通用 Euler、Heun、RK45 通常将整个右端视为待积分函数。

而 DPM-Solver 将问题拆成：

$$
\boxed{
\text{解析求解线性项}
+
\text{数值近似神经网络积分}
}
\tag{1.2}
$$

因此，它属于利用问题结构的**指数积分方法（Exponential Integrator）**。

原论文主要贡献包括：

- 推导扩散 PF ODE 的精确积分表达式。
- 使用半对数信噪比 $\lambda$ 简化积分。
- 构造一阶、二阶、三阶求解器。
- 给出相应收敛阶分析。
- 研究固定步长、自适应步长和不同噪声调度。
- 证明确定性 DDIM 可以视为一阶 DPM-Solver。
- 支持连续时间和离散时间训练的扩散模型。

这些内容分别对应原论文 Section 2–5 及附录。[arXiv](https://arxiv.org/html/2206.00927?utm_source=chatgpt.com)

## 2. 数学基础：高斯边缘分布与扩散 SDE

### 2.1 高斯边缘分布

定义真实数据：

$$
x_0\sim p_{\mathrm{data}}.
$$

扩散模型的前向边缘分布满足：

$$
\boxed{
q(x_t\mid x_0)
=
\mathcal N(\alpha_t x_0,\sigma_t^2I)
}
\tag{2.1}
$$

等价地：

$$
\boxed{
x_t=\alpha_t x_0+\sigma_t\epsilon,
\qquad
\epsilon\sim\mathcal N(0,I)
}
\tag{2.2}
$$

其中：

- $\alpha_t>0$：信号系数；
- $\sigma_t>0$：噪声标准差；
- $t\in(0,T]$：扩散时间。

定义信噪比：

$$
\boxed{
\mathrm{SNR}(t)
=
\frac{\alpha_t^2}{\sigma_t^2}
}
\tag{2.3}
$$

论文假设它随正向时间严格递减。

特别注意：

$$
\alpha_t^2+\sigma_t^2=1
$$

是 VP 调度的性质，**不是 DPM-Solver 对所有噪声调度的通用要求**。

### 2.2 对应的线性 SDE

考虑：

$$
\boxed{
dx_t=f(t)x_t\,dt+g(t)dW_t
}
\tag{2.4}
$$

要使它具有式（2.1）的边缘分布，系数需要满足：

$$
\boxed{
f(t)=\frac{d\log\alpha_t}{dt}
}
\tag{2.5}
$$

以及：

$$
\boxed{
g^2(t)
=
\frac{d\sigma_t^2}{dt}
-
2f(t)\sigma_t^2
}
\tag{2.6}
$$

第二个公式可以从线性 SDE 的方差演化推导。

对：

$$
dx_t=f(t)x_tdt+g(t)dW_t
$$

应用 Itô 公式：

$$
d(x_t^2)
=
2x_tdx_t+(dx_t)^2.
$$

代入：

$$
d(x_t^2)
=
2f(t)x_t^2dt
+
2g(t)x_tdW_t
+
g^2(t)dt.
$$

取期望，随机积分项的期望为零：

$$
\frac{d}{dt}\mathbb E[x_t^2]
=
2f(t)\mathbb E[x_t^2]+g^2(t).
$$

对于给定 $x_0$ 的条件方差：

$$
\operatorname{Var}(x_t\mid x_0)=\sigma_t^2,
$$

得到：

$$
\frac{d\sigma_t^2}{dt}
=
2f(t)\sigma_t^2+g^2(t).
$$

整理即式（2.6）。

### 2.3 从 SDE 到 PF ODE

正向 SDE：

$$
dx_t=f(t)x_tdt+g(t)dW_t
$$

对应的 Probability Flow ODE 为：

$$
\boxed{
\frac{dx_t}{dt}
=
f(t)x_t
-
\frac12g^2(t)\nabla_x\log p_t(x_t)
}
\tag{2.7}
$$

其中：

$$
\nabla_x\log p_t(x)
$$

是边缘分布的 Score。

根据高斯加噪关系：

$$
\boxed{
\epsilon^*(x,t)
=
-\sigma_t\nabla_x\log p_t(x)
}
\tag{2.8}
$$

使用神经网络近似：

$$
\epsilon_\theta(x,t)\approx\epsilon^*(x,t),
$$

得到：

$$
\boxed{
\frac{dx_t}{dt}
=
f(t)x_t+
\frac{g^2(t)}{2\sigma_t}
\epsilon_\theta(x_t,t)
}
\tag{2.9}
$$

这就是 DPM-Solver 的基本 ODE。

需要强调：这里的 $\epsilon^*$ 是给定 $x_t$ 后噪声的条件期望，而逐样本噪声监督标签是式（2.2）中的 $\epsilon$。两者不能混淆。

## 3. 精确积分公式：论文最重要的数学推导

### 3.1 为什么不能直接使用通用 ODE 求解器？

式（2.9）可以写成：

$$
\frac{dx_t}{dt}
=
f(t)x_t+
b(t)\epsilon_\theta(x_t,t),
$$

其中：

$$
b(t)=\frac{g^2(t)}{2\sigma_t}.
$$

如果使用 Euler：

$$
x_{t+h}
\approx
x_t+h
\left[
f(t)x_t+b(t)\epsilon_\theta(x_t,t)
\right],
$$

那么即使线性项：

$$
f(t)x_t
$$

可以解析求解，Euler 仍然对其进行近似。

DPM-Solver 的思路是先将线性项消除。

### 3.2 积分因子

由于：

$$
f(t)=\frac{\dot\alpha_t}{\alpha_t},
$$

式（2.9）变成：

$$
\frac{dx_t}{dt}
=
\frac{\dot\alpha_t}{\alpha_t}x_t
+
\frac{g^2(t)}{2\sigma_t}\epsilon_\theta.
$$

两边除以 $\alpha_t$：

$$
\frac1{\alpha_t}\frac{dx_t}{dt}
=
\frac{\dot\alpha_t}{\alpha_t^2}x_t
+
\frac{g^2(t)}{2\alpha_t\sigma_t}\epsilon_\theta.
$$

将第一项移到左边：

$$
\frac1{\alpha_t}\frac{dx_t}{dt}
-
\frac{\dot\alpha_t}{\alpha_t^2}x_t
=
\frac{g^2(t)}{2\alpha_t\sigma_t}\epsilon_\theta.
$$

左边正好是：

$$
\frac{d}{dt}\left(\frac{x_t}{\alpha_t}\right).
$$

因此：

$$
\boxed{
\frac{d}{dt}
\left(
\frac{x_t}{\alpha_t}
\right)
=
\frac{g^2(t)}{2\alpha_t\sigma_t}
\epsilon_\theta(x_t,t)
}
\tag{3.1}
$$

现在已知线性项已经被精确处理。

### 3.3 为什么使用半对数信噪比？

定义：

$$
\boxed{
\lambda_t
=
\log\frac{\alpha_t}{\sigma_t}
}
\tag{3.2}
$$

它等于：

$$
\lambda_t
=
\frac12\log\mathrm{SNR}(t).
$$

对时间求导：

$$
\dot\lambda_t
=
\frac{\dot\alpha_t}{\alpha_t}
-
\frac{\dot\sigma_t}{\sigma_t}.
$$

根据式（2.6）：

$$
\begin{aligned}
g^2(t)
&=
2\sigma_t\dot\sigma_t
-
2\frac{\dot\alpha_t}{\alpha_t}\sigma_t^2\\
&=
2\sigma_t^2
\left(
\frac{\dot\sigma_t}{\sigma_t}
-
\frac{\dot\alpha_t}{\alpha_t}
\right).
\end{aligned}
$$

所以：

$$
\boxed{
g^2(t)
=
-2\sigma_t^2\dot\lambda_t
}
\tag{3.3}
$$

将它代入式（3.1）：

$$
\begin{aligned}
\frac{d}{dt}
\left(\frac{x_t}{\alpha_t}\right)
&=
-\frac{\sigma_t}{\alpha_t}
\dot\lambda_t\epsilon_\theta\\
&=
-e^{-\lambda_t}
\dot\lambda_t\epsilon_\theta.
\end{aligned}
$$

因此：

$$
\boxed{
\frac{d}{d\lambda}
\left(
\frac{x_\lambda}{\alpha_\lambda}
\right)
=
-e^{-\lambda}\epsilon_\theta(x_\lambda,\lambda)
}
\tag{3.4}
$$

这个变量替换的意义非常重要。

原来的积分系数：

$$
\frac{g^2(t)}{2\alpha_t\sigma_t}
$$

变成了：

$$
-e^{-\lambda}.
$$

复杂的噪声调度系数被吸收进时间变换。

### 3.4 精确积分表达式

考虑从 $s$ 采样到 $t$，其中：

$$
t<s,\qquad \lambda_t>\lambda_s.
$$

对式（3.4）积分：

$$
\frac{x_t}{\alpha_t}
-
\frac{x_s}{\alpha_s}
=
-\int_{\lambda_s}^{\lambda_t}
e^{-\lambda}
\epsilon_\theta(x_\lambda,\lambda)\,d\lambda.
$$

得到：

$$
\boxed{
x_t
=
\frac{\alpha_t}{\alpha_s}x_s
-
\alpha_t
\int_{\lambda_s}^{\lambda_t}
e^{-\lambda}
\epsilon_\theta(x_\lambda,\lambda)\,d\lambda
}
\tag{3.5}
$$

这对应原论文 Proposition 3.1。

注意：

**它是精确的积分恒等式，但还不是可以直接计算的采样更新公式。**

因为积分中的：

$$
\epsilon_\theta(x_\lambda,\lambda)
$$

依赖尚未求出的中间轨迹。

DPM-Solver 后续算法的任务，就是高效近似这一积分。[arXiv](https://arxiv.org/html/2206.00927?utm_source=chatgpt.com)

---

## 4. DPM-Solver-1：从精确积分到 DDIM

令：

$$
h=\lambda_t-\lambda_s>0.
$$

记沿 ODE 轨迹的网络输出：

$$
E(\lambda)=\epsilon_\theta(x_\lambda,\lambda).
$$

精确积分为：

$$
x_t
=
\frac{\alpha_t}{\alpha_s}x_s
-
\alpha_t
\int_{\lambda_s}^{\lambda_t}
e^{-\lambda}E(\lambda)\,d\lambda.
$$

一阶方法在当前区间使用常数近似：

$$
E(\lambda)\approx E(\lambda_s)=E_s.
$$

于是：

$$
\begin{aligned}
\int_{\lambda_s}^{\lambda_t}e^{-\lambda}E(\lambda)d\lambda
&\approx
E_s\left(e^{-\lambda_s}-e^{-\lambda_t}\right).
\end{aligned}
$$

代入：

$$
x_t
=
\frac{\alpha_t}{\alpha_s}x_s
-
\alpha_t
\left(e^{-\lambda_s}-e^{-\lambda_t}\right)E_s.
$$

利用：

$$
\alpha_t e^{-\lambda_t}=\sigma_t
$$

以及：

$$
\alpha_t e^{-\lambda_s}=\sigma_t e^h,
$$

得到：

$$
\boxed{
x_t
=
\frac{\alpha_t}{\alpha_s}x_s
-
\sigma_t(e^h-1)\epsilon_\theta(x_s,s)
}
\tag{4.1}
$$

这就是 DPM-Solver-1。

每步仅需一次网络预测。

### 4.1 与 DDIM 的关系

确定性 DDIM 可以写成：

$$
\boxed{
x_t=\alpha_t\hat x_0+\sigma_t\hat\epsilon
}
\tag{4.2}
$$

其中：

$$
\hat\epsilon=\epsilon_\theta(x_s,s),
$$

$$
\hat x_0
=
\frac{x_s-\sigma_s\hat\epsilon}{\alpha_s}.
$$

代入：

$$
\begin{aligned}
x_t
&=
\frac{\alpha_t}{\alpha_s}x_s
-
\frac{\alpha_t\sigma_s}{\alpha_s}\hat\epsilon
+
\sigma_t\hat\epsilon\\
&=
\frac{\alpha_t}{\alpha_s}x_s
-
\left(
\frac{\alpha_t\sigma_s}{\alpha_s}-\sigma_t
\right)\hat\epsilon.
\end{aligned}
$$

又因为：

$$
\frac{\alpha_t\sigma_s}{\alpha_s}
=
\sigma_t e^h,
$$

所以：

$$
\boxed{
x_t
=
\frac{\alpha_t}{\alpha_s}x_s
-
\sigma_t(e^h-1)\hat\epsilon
}
\tag{4.3}
$$

与式（4.1）完全一致。

因此：

$$
\boxed{
\text{DPM-Solver-1}
\equiv
\text{DDIM}(\eta=0)
}
\tag{4.4}
$$

这里的等价是指相同噪声调度、相同时间节点和一致的模型预测转换下，更新公式完全相同。

这个结论也解释了 DDIM 为什么能够跳步：它本身就是一个确定性 ODE 的一阶积分更新，而不是必须逐步执行原始 DDPM 马尔可夫反向链。

## 5. DPM-Solver-2：二阶指数积分

### 5.1 二阶方法究竟近似什么？

继续考虑：

$$
I=
\int_{\lambda_s}^{\lambda_t}
e^{-\lambda}E(\lambda)d\lambda.
$$

一阶方法假设：

$$
E(\lambda)\approx E_s.
$$

二阶方法进一步考虑网络输出沿 ODE 轨迹的变化：

$$
\boxed{
E(\lambda)
=
E_s+
(\lambda-\lambda_s)E_s'
+
O((\lambda-\lambda_s)^2)
}
\tag{5.1}
$$

这里：

$$
E_s'
=
\frac{d}{d\lambda}
\epsilon_\theta(x_\lambda,\lambda)
\bigg|_{\lambda=\lambda_s}
$$

是全导数，既包括显式时间依赖，也包括状态变化引起的网络输出变化。

代入积分：

$$
I
\approx
E_sI_0+E_s'I_1,
$$

其中：

$$
I_0
=
\int_{\lambda_s}^{\lambda_t}e^{-\lambda}d\lambda
=
e^{-\lambda_s}(1-e^{-h}),
$$

$$
I_1
=
\int_{\lambda_s}^{\lambda_t}
e^{-\lambda}(\lambda-\lambda_s)d\lambda.
$$

令：

$$
u=\lambda-\lambda_s,
$$

则：

$$
I_1=e^{-\lambda_s}\int_0^h ue^{-u}du.
$$

利用：

$$
\int ue^{-u}du=-(u+1)e^{-u},
$$

得到：

$$
\boxed{
I_1
=
e^{-\lambda_s}
\left[1-(h+1)e^{-h}\right]
}
\tag{5.2}
$$

因此：

$$
\boxed{
\begin{aligned}
x_t
&=
\frac{\alpha_t}{\alpha_s}x_s
-
\sigma_t(e^h-1)E_s\\
&\quad-
\sigma_t(e^h-1-h)E_s'
+
O(h^3).
\end{aligned}
}
\tag{5.3}
$$

这一公式说明：只要能够估计 $E_s'$，就可以构造二阶方法。

### 5.2 为什么使用中间点？

直接计算：

$$
E_s'
$$

需要沿 ODE 轨迹对神经网络求导。

这涉及：

$$
\frac{\partial\epsilon_\theta}{\partial x}
\frac{dx}{d\lambda}
+
\frac{\partial\epsilon_\theta}{\partial\lambda}.
$$

对于大型神经网络，这种方式计算开销较高。

因此论文使用中间点的网络输出估计变化率。

定义：

$$
\lambda_m=\lambda_s+\frac h2.
$$

首先使用一阶方法估计中间状态：

$$
\boxed{
u
=
\frac{\alpha_m}{\alpha_s}x_s
-
\sigma_m(e^{h/2}-1)E_s
}
\tag{5.4}
$$

再计算：

$$
E_m=\epsilon_\theta(u,m).
$$

由于：

$$
E_m
=
E_s+\frac h2E_s'+O(h^2),
$$

因此：

$$
\boxed{
E_s'
=
\frac{2(E_m-E_s)}h+O(h)
}
\tag{5.5}
$$

将其代入式（5.3），就得到一个二阶指数积分方法。

### 5.3 原论文 Algorithm 1 的具体更新

需要特别区分：

- 精确保留指数积分一阶矩得到的二阶公式；
- 原论文 Algorithm 1 采用的指数中点型二阶公式。

原论文的 DPM-Solver-2 使用：

$$
\boxed{
\begin{aligned}
u
&=
\frac{\alpha_m}{\alpha_s}x_s
-
\sigma_m(e^{h/2}-1)
\epsilon_\theta(x_s,s),\\[4pt]
x_t
&=
\frac{\alpha_t}{\alpha_s}x_s
-
\sigma_t(e^h-1)
\epsilon_\theta(u,m).
\end{aligned}
}
\tag{5.6}
$$

这与前面的 Taylor 展开并不矛盾。

因为：

$$
E_m=E_s+\frac h2E_s'+O(h^2)
$$

所以：

$$
\begin{aligned}
&-\sigma_t(e^h-1)E_m\\
&=
-\sigma_t(e^h-1)E_s
-\sigma_t\frac h2(e^h-1)E_s'
+O(h^3).
\end{aligned}
$$

而：

$$
\frac h2(e^h-1)
=
\frac{h^2}{2}+O(h^3),
$$

$$
e^h-1-h
=
\frac{h^2}{2}+O(h^3).
$$

因此两种方法的二阶项一致，差异出现在更高阶项。

这也是为什么不能把任意一个二阶指数积分公式直接称为原论文 Algorithm 1。

**原论文 Algorithm 1 的关键特点是：用中间点网络预测替代整段积分中的网络预测。** [arXiv](https://arxiv.org/html/2206.00927?utm_source=chatgpt.com)

### 5.4 与 Heun 的区别

Heun：

$$
\tilde x_t=x_s+\Delta t\,v_\theta(x_s,s),
$$

$$
x_t
=
x_s+
\frac{\Delta t}{2}
\left[
v_\theta(x_s,s)+v_\theta(\tilde x_t,t)
\right].
$$

DPM-Solver-2：

$$
u=
\frac{\alpha_m}{\alpha_s}x_s
-
\sigma_m(e^{h/2}-1)E_s,
$$

$$
x_t=
\frac{\alpha_t}{\alpha_s}x_s
-
\sigma_t(e^h-1)E_m.
$$

两者都是二阶方法，但区别在于：

| 项目 | Heun | DPM-Solver-2 |
|---|---|---|
| 数学基础 | 通用 Runge–Kutta | 指数积分 |
| 已知线性部分 | 数值近似 | 解析处理 |
| 时间变量 | 通常为 $t$ | 半对数信噪比 $\lambda$ |
| 网络调用 | 每步 2 次 | 每步 2 次 |
| 积分对象 | 完整速度场 | 指数加权网络预测 |

因此，DPM-Solver-2 的优势并不是 NFE 更少，而是在相同 NFE 下能够利用扩散 ODE 的特殊结构降低某些离散化误差。

## 6. DPM-Solver-3：三阶方法

二阶方法使用：

$$
E(\lambda)
=
E_s+
(\lambda-\lambda_s)E_s'
+
O(h^2).
$$

三阶方法进一步保留：

$$
\boxed{
E(\lambda)
=
E_s+
(\lambda-\lambda_s)E_s'
+
\frac12(\lambda-\lambda_s)^2E_s''
+
O(h^3)
}
\tag{6.1}
$$

代入：

$$
I=
\int_{\lambda_s}^{\lambda_t}
e^{-\lambda}E(\lambda)d\lambda
$$

得到：

$$
\boxed{
I
=
I_0E_s+I_1E_s'
+\frac12I_2E_s''
+O(h^4)
}
\tag{6.2}
$$

其中：

$$
I_2=
\int_{\lambda_s}^{\lambda_t}
e^{-\lambda}(\lambda-\lambda_s)^2d\lambda.
$$

令：

$$
u=\lambda-\lambda_s,
$$

则：

$$
I_2=e^{-\lambda_s}\int_0^h u^2e^{-u}du.
$$

分部积分得到：

$$
\boxed{
I_2
=
e^{-\lambda_s}
\left[
2-(h^2+2h+2)e^{-h}
\right]
}
\tag{6.3}
$$

因此三阶方法需要估计：

$$
E_s',\qquad E_s''.
$$

论文通过两个中间阶段的网络预测构造这些近似，而不是直接对神经网络进行二阶自动微分。

### 6.1 三阶方法为什么需要三次网络调用？

第一阶段：

$$
E_s=\epsilon_\theta(x_s,s).
$$

第二阶段：

在第一个中间点预测：

$$
E_{m_1}.
$$

第三阶段：

在第二个中间点预测：

$$
E_{m_2}.
$$

利用三个函数值估计网络预测的局部变化。

因此：

$$
\boxed{\mathrm{NFE}=3}
$$

三阶方法在适当的光滑性和步长条件下具有：

$$
\boxed{
\text{局部截断误差}=O(h^4)
}
$$

$$
\boxed{
\text{全局误差}=O(h_{\max}^3)
}
$$

但需要注意：**高阶方法不一定在任意 NFE、任意模型误差下都更好。**

当步长很大、模型预测不够平滑或引导强度过高时，较高阶的局部近似可能不稳定。

原论文给出了具体的 DPM-Solver-3 算法及收敛证明，不能仅凭 Taylor 展开式（6.2）就认定任意三点实现具有三阶精度。

## 7. 收敛阶：为什么是 $O(h_{\max}^k)$？

原论文 Theorem 3.2 证明，在适当的正则性条件下，DPM-Solver-$k$ 对于：

$$
k=1,2,3
$$

具有：

$$
\boxed{
\|\tilde x_0-x_0^{\mathrm{ODE}}\|
=
O(h_{\max}^k)
}
\tag{7.1}
$$

其中：

- $\tilde x_0$：数值求解器得到的终点；
- $x_0^{\mathrm{ODE}}$：参数化 ODE 的精确终点；
- $h_{\max}$：所有半对数信噪比步长中的最大值。

这里的误差是：

$$
\boxed{
\text{数值求解误差}
}
$$

而不是：

$$
\boxed{
\text{生成结果与真实数据之间的误差}
}
$$

这一区别非常重要。

假设神经网络预测本身存在偏差：

$$
\epsilon_\theta\neq\epsilon^*.
$$

即使使用无限精确的 ODE 求解器，最终生成分布仍然可能偏离真实数据分布。

因此可以将误差来源概念性地拆分为：

$$
\boxed{
\text{生成误差}
\longleftarrow
\text{模型误差}
+
\text{数值积分误差}
}
$$

DPM-Solver 主要降低第二项。

### 7.1 局部误差如何累积为全局误差？

对于 $k$ 阶方法：

$$
\text{局部截断误差}=O(h^{k+1}).
$$

假设积分区间总长度固定，并使用近似均匀步长：

$$
h\approx\frac{L}{M}.
$$

则需要：

$$
M\approx\frac Lh
$$

步。

如果误差传播稳定，局部误差累计量级约为：

$$
M O(h^{k+1})
=
O(h^k).
$$

因此：

$$
\boxed{
\text{全局误差}=O(h^k)
}
\tag{7.2}
$$

但这依赖稳定性、光滑性等条件，并非只要写出一个 $k$ 阶 Taylor 展开就自动成立。[arXiv](https://arxiv.org/html/2206.00927?utm_source=chatgpt.com)

## 8. 时间步长设计：为什么在 logSNR 空间均匀采样？

DPM-Solver 的时间变量是：

$$
\lambda_t=\log\frac{\alpha_t}{\sigma_t}.
$$

因此最自然的固定步长方案是：

$$
\boxed{
\lambda_i
=
\lambda_T+
\frac{i}{M}
(\lambda_\epsilon-\lambda_T),
\qquad i=0,\ldots,M
}
\tag{8.1}
$$

其中 $\epsilon>0$ 是实际使用的最小扩散时间，以避免某些调度在 $t=0$ 附近出现奇异性。

然后通过反函数：

$$
\boxed{
t_i=\lambda^{-1}(\lambda_i)
}
\tag{8.2}
$$

得到原始扩散时间节点。

### 8.1 为什么不直接均匀划分 $t$？

因为：

$$
\lambda(t)
$$

通常不是 $t$ 的线性函数。

如果：

$$
t_i=T-\frac{iT}{M},
$$

那么对应的：

$$
\lambda_{i+1}-\lambda_i
$$

可能差异很大。

而 DPM-Solver 的 Taylor 展开和误差分析是在：

$$
\lambda
$$

空间进行的。

因此均匀划分 $\lambda$ 更符合其数值方法的数学结构。

但这不意味着均匀 logSNR 网格在所有模型上都严格最优。

### 8.2 自适应步长

原论文还提出自适应方案。

其基本思想是同时构造不同阶数的数值估计，例如：

$$
x_{i+1}^{(2)}
$$

和：

$$
x_{i+1}^{(3)}.
$$

通过两者差异估计局部误差：

$$
\boxed{
e_i
\approx
\|x_{i+1}^{(3)}-x_{i+1}^{(2)}\|
}
\tag{8.3}
$$

然后：

- 误差较大时缩小步长。
- 误差较小时增大步长。
- 超过容差时拒绝当前步并重新计算。

论文对少 NFE 场景主要使用固定步长组合策略，而对更大 NFE 预算使用自适应策略。[arXiv](https://arxiv.org/html/2206.00927?utm_source=chatgpt.com)

## 9. 连续时间与离散时间模型

DPM-Solver 不要求网络必须采用连续时间训练。

假设原始 DDPM 在离散时间：

$$
n=0,1,\ldots,N-1
$$

上训练噪声预测模型：

$$
\epsilon_\theta(x,n).
$$

DPM-Solver 可以通过时间映射，将连续采样时间对应到模型的时间条件输入。

例如：

$$
\boxed{
n(t)=\frac{N-1}{T}t
}
\tag{9.1}
$$

然后：

$$
\boxed{
\epsilon_\theta^{\mathrm{cont}}(x,t)
=
\epsilon_\theta(x,n(t))
}
\tag{9.2}
$$

即使：

$$
n(t)
$$

不是整数，具有连续时间嵌入结构的网络仍可能支持这种调用。

不过实际应用中，需要同时处理：

- 训练时使用的时间索引约定；
- 离散噪声调度的插值；
- $\alpha_t,\sigma_t$ 的连续化；
- 网络对非整数时间输入的支持。

因此这不是简单地把整数时间强制转换为浮点数就一定有效。

## 10. 原论文实验与结论

原论文主要比较：

- DDIM；
- Euler–Maruyama；
- 通用 RK 求解器；
- DPM-Solver 不同阶数；
- 固定和自适应时间步长。

在 CIFAR-10 的连续时间模型上，论文报告：

| NFE | DPM-Solver FID |
|---:|---:|
| 10 | 4.70 |
| 12 | 3.75 |
| 15 | 3.24 |
| 20 | 2.87 |

实验说明：对于所测试的模型，在少步数采样场景下，DPM-Solver 相比通用 RK 方法具有明显优势。

但这些结果依赖具体预训练模型、数据集和实验设置，不能直接作为所有扩散模型的性能保证。[arXiv](https://arxiv.org/html/2206.00927?utm_source=chatgpt.com)

---

# 第二部分：DPM-Solver++ 原论文

## 11. 为什么需要 DPM-Solver++？

DPM-Solver++ 研究的是一个更具体的问题：

$$
\boxed{
\text{强引导条件下的少步数扩散采样}
}
$$

例如 Classifier-Free Guidance：

$$
\boxed{
\tilde\epsilon_\theta
=
w\epsilon_\theta(x,t,c)
+
(1-w)\epsilon_\theta(x,t,\varnothing)
}
\tag{11.1}
$$

其中：

- $c$：生成条件；
- $\varnothing$：无条件输入；
- $w$：引导强度。

当：

$$
w>1
$$

时，模型预测被向条件方向外推。

较大的 $w$ 可能放大网络输出及其导数。

这会导致两个问题。

**问题一：高阶求解器不稳定。**

高阶方法依赖网络预测在积分区间内的平滑变化。

强引导可能使网络输出随状态变化更加剧烈，从而要求更小的积分步长。

**问题二：生成结果可能超出训练数据的典型范围。**

对于归一化图像：

$$
x_0\in[-1,1]^D,
$$

强引导可能使预测结果超出合理范围，导致过度饱和等问题。

DPM-Solver++ 因此提出三项改进：

$$
\boxed{
\text{数据预测参数化}
+
\text{Thresholding}
+
\text{Multistep}
}
\tag{11.2}
$$

这些是原论文 Section 3–4 的核心内容。[arxiv.org](https://arxiv.org/pdf/2211.01095?utm_source=chatgpt.com)

## 12. 从噪声预测转换为数据预测

根据：

$$
x_t=\alpha_t x_0+\sigma_t\epsilon,
$$

定义数据预测模型：

$$
\boxed{
D_\theta(x_t,t)
=
\frac{x_t-\sigma_t\epsilon_\theta(x_t,t)}
{\alpha_t}
}
\tag{12.1}
$$

反过来：

$$
\boxed{
\epsilon_\theta(x_t,t)
=
\frac{x_t-\alpha_tD_\theta(x_t,t)}
{\sigma_t}
}
\tag{12.2}
$$

需要注意：

**DPM-Solver++ 不要求重新训练一个数据预测网络。**

如果原模型预测噪声，可以直接使用式（12.1）转换成数据预测。

因此它仍然属于 training-free sampler。

## 13. 推导数据预测形式的精确积分

从 PF ODE：

$$
\frac{dx_t}{dt}
=
f(t)x_t+
\frac{g^2(t)}{2\sigma_t}
\epsilon_\theta(x_t,t)
$$

出发。

代入：

$$
\epsilon_\theta
=
\frac{x_t-\alpha_tD_\theta}{\sigma_t}.
$$

得到：

$$
\frac{dx_t}{dt}
=
f(t)x_t+
\frac{g^2(t)}{2\sigma_t^2}
\left(x_t-\alpha_tD_\theta\right).
$$

整理：

$$
\boxed{
\frac{dx_t}{dt}
=
\left(
f(t)+\frac{g^2(t)}{2\sigma_t^2}
\right)x_t
-
\frac{\alpha_tg^2(t)}{2\sigma_t^2}
D_\theta(x_t,t)
}
\tag{13.1}
$$

根据：

$$
g^2(t)
=
2\sigma_t\dot\sigma_t
-
2f(t)\sigma_t^2,
$$

有：

$$
\frac{g^2(t)}{2\sigma_t^2}
=
\frac{\dot\sigma_t}{\sigma_t}-f(t).
$$

因此：

$$
f(t)+\frac{g^2(t)}{2\sigma_t^2}
=
\frac{\dot\sigma_t}{\sigma_t}.
$$

同时：

$$
-\frac{\alpha_tg^2(t)}{2\sigma_t^2}
=
\alpha_t\dot\lambda_t.
$$

所以：

$$
\boxed{
\frac{dx_t}{dt}
=
\frac{\dot\sigma_t}{\sigma_t}x_t
+
\alpha_t\dot\lambda_tD_\theta(x_t,t)
}
\tag{13.2}
$$

现在使用积分因子：

$$
\frac1{\sigma_t}.
$$

两边整理：

$$
\boxed{
\frac{d}{dt}
\left(
\frac{x_t}{\sigma_t}
\right)
=
\frac{\alpha_t}{\sigma_t}
\dot\lambda_tD_\theta(x_t,t)
}
\tag{13.3}
$$

因为：

$$
\frac{\alpha_t}{\sigma_t}=e^{\lambda_t},
$$

所以：

$$
\boxed{
\frac{d}{d\lambda}
\left(
\frac{x_\lambda}{\sigma_\lambda}
\right)
=
e^\lambda D_\theta(x_\lambda,\lambda)
}
\tag{13.4}
$$

积分得到：

$$
\boxed{
x_t
=
\frac{\sigma_t}{\sigma_s}x_s
+
\sigma_t
\int_{\lambda_s}^{\lambda_t}
e^\lambda D_\theta(x_\lambda,\lambda)d\lambda
}
\tag{13.5}
$$

这对应 DPM-Solver++ Proposition 4.1。

与 DPM-Solver 对比：

$$
\boxed{
\begin{aligned}
\text{DPM-Solver:}\quad
x_t
&=
\frac{\alpha_t}{\alpha_s}x_s
-
\alpha_t
\int e^{-\lambda}\epsilon_\theta\,d\lambda,\\[4pt]
\text{DPM-Solver++:}\quad
x_t
&=
\frac{\sigma_t}{\sigma_s}x_s
+
\sigma_t
\int e^\lambda D_\theta\,d\lambda.
\end{aligned}
}
\tag{13.6}
$$

这是两篇论文最重要的数学联系。

两种精确积分表达式描述同一个 ODE，但它们对不同函数进行数值近似，因此有限步长下的误差表现可以不同。[arXiv](https://arxiv.org/pdf/2211.01095?utm_source=chatgpt.com)

---

## 14. DPM-Solver++(2S)：二阶单步方法

### 14.1 一阶数据预测更新

从式（13.5）出发：

$$
x_t
=
\frac{\sigma_t}{\sigma_s}x_s
+
\sigma_t
\int_{\lambda_s}^{\lambda_t}
e^\lambda D_\theta(x_\lambda,\lambda)d\lambda.
$$

假设在当前区间：

$$
D_\theta(x_\lambda,\lambda)\approx D_s,
$$

其中：

$$
D_s=D_\theta(x_s,s).
$$

则：

$$
\begin{aligned}
x_t
&\approx
\frac{\sigma_t}{\sigma_s}x_s
+
\sigma_tD_s(e^{\lambda_t}-e^{\lambda_s})\\
&=
\frac{\sigma_t}{\sigma_s}x_s
+
\alpha_t(1-e^{-h})D_s.
\end{aligned}
$$

因此：

$$
\boxed{
x_t
=
\frac{\sigma_t}{\sigma_s}x_s
+
\alpha_t(1-e^{-h})D_s
}
\tag{14.1}
$$

这是 DPM-Solver++ 的一阶更新，也与确定性 DDIM 等价。

### 14.2 二阶方法需要什么？

令：

$$
D(\lambda)=D_\theta(x_\lambda,\lambda).
$$

在 $\lambda_s$ 附近展开：

$$
D(\lambda)
=
D_s+(\lambda-\lambda_s)D_s'+O(h^2).
$$

代入：

$$
I=
\int_{\lambda_s}^{\lambda_t}
e^\lambda D(\lambda)d\lambda.
$$

得到：

$$
I
\approx
e^{\lambda_s}
\left[
(e^h-1)D_s+
((h-1)e^h+1)D_s'
\right].
$$

因此：

$$
\boxed{
\begin{aligned}
x_t
&=
\frac{\sigma_t}{\sigma_s}x_s
+
\alpha_t(1-e^{-h})D_s\\
&\quad+
\alpha_t(h-1+e^{-h})D_s'
+
O(h^3).
\end{aligned}
}
\tag{14.2}
$$

为了估计：

$$
D_s',
$$

使用中间点。

### 14.3 原论文 Algorithm 1

选择：

$$
\lambda_m=\lambda_s+rh,\qquad 0<r<1.
$$

先使用一阶方法预测中间状态：

$$
\boxed{
u
=
\frac{\sigma_m}{\sigma_s}x_s
+
\alpha_m(1-e^{-rh})D_s
}
\tag{14.3}
$$

然后计算：

$$
D_m=D_\theta(u,m).
$$

定义组合预测：

$$
\boxed{
\bar D
=
\left(1-\frac1{2r}\right)D_s
+
\frac1{2r}D_m
}
\tag{14.4}
$$

最后更新：

$$
\boxed{
x_t
=
\frac{\sigma_t}{\sigma_s}x_s
+
\alpha_t(1-e^{-h})\bar D
}
\tag{14.5}
$$

这就是论文 Algorithm 1 的 DPM-Solver++(2S)。

特别地，当：

$$
r=\frac12,
$$

有：

$$
1-\frac1{2r}=0,
\qquad
\frac1{2r}=1.
$$

因此：

$$
\bar D=D_m.
$$

更新简化为：

$$
\boxed{
x_t
=
\frac{\sigma_t}{\sigma_s}x_s
+
\alpha_t(1-e^{-h})D_m
}
\tag{14.6}
$$

这与原始 DPM-Solver-2 的指数中点思想对应，只不过现在积分对象是数据预测，而不是噪声预测。

### 14.4 为什么是二阶？

根据：

$$
D_m
=
D_s+rhD_s'+O(h^2),
$$

代入式（14.4）：

$$
\begin{aligned}
\bar D
&=
\left(1-\frac1{2r}\right)D_s
+
\frac1{2r}
\left(D_s+rhD_s'+O(h^2)\right)\\
&=
D_s+\frac h2D_s'+O(h^2).
\end{aligned}
$$

再代入式（14.5）：

$$
\begin{aligned}
x_t
&=
\frac{\sigma_t}{\sigma_s}x_s\\
&\quad+
\alpha_t(1-e^{-h})
\left(D_s+\frac h2D_s'+O(h^2)\right).
\end{aligned}
$$

由于：

$$
\frac h2(1-e^{-h})
=
\frac{h^2}{2}+O(h^3),
$$

而：

$$
h-1+e^{-h}
=
\frac{h^2}{2}+O(h^3),
$$

所以式（14.5）与精确积分展开式（14.2）的前两阶一致。

因此它是二阶方法。

这里要特别注意：**原论文 2S 使用的是简化的二阶系数，不是必须精确保留指数积分的一阶矩才能达到二阶。**

---

## 15. DPM-Solver++(2M)：二阶多步方法

这是最值得深入理解的部分，因为它解释了为什么能够利用历史网络预测，在每个新时间节点只调用一次网络。

### 15.1 单步与多步的区别

2S 在当前积分区间中使用：

$$
D_s,\quad D_m.
$$

因此每步需要两次网络推理。

2M 则使用：

$$
D_{i-1},\quad D_i,
$$

其中前一个网络预测已经计算过。

于是无需再构造中间点。

这种思想与经典数值分析中的 Adams–Bashforth 多步方法类似。

但 DPM-Solver++(2M) 仍然保留了扩散 ODE 的指数积分结构。

### 15.2 时间与步长定义

设反向采样节点为：

$$
t_0>t_1>\cdots>t_M.
$$

对应：

$$
\lambda_0<\lambda_1<\cdots<\lambda_M.
$$

定义：

$$
\boxed{
h_i=\lambda_i-\lambda_{i-1}>0
}
\tag{15.1}
$$

以及：

$$
\boxed{
r_i=\frac{h_{i-1}}{h_i}
}
\tag{15.2}
$$

令：

$$
D_{i-1}=D_\theta(x_{i-1},t_{i-1}),
$$

$$
D_{i-2}=D_\theta(x_{i-2},t_{i-2}).
$$

在当前节点：

$$
\lambda_{i-1}
$$

附近，网络预测的一阶导数可以用历史差分近似：

$$
\boxed{
D'_{i-1}
\approx
\frac{D_{i-1}-D_{i-2}}{h_{i-1}}
}
\tag{15.3}
$$

### 15.3 历史差分如何得到当前区间的预测？

对于当前区间：

$$
[\lambda_{i-1},\lambda_i],
$$

有：

$$
D(\lambda)
\approx
D_{i-1}
+
(\lambda-\lambda_{i-1})D'_{i-1}.
$$

如果只考虑区间平均意义下的一阶变化：

$$
\bar D
\approx
D_{i-1}+\frac{h_i}{2}D'_{i-1}.
$$

代入历史差分：

$$
\bar D
\approx
D_{i-1}
+
\frac{h_i}{2h_{i-1}}
(D_{i-1}-D_{i-2}).
$$

因为：

$$
r_i=\frac{h_{i-1}}{h_i},
$$

所以：

$$
\boxed{
\bar D
=
D_{i-1}
+
\frac1{2r_i}
(D_{i-1}-D_{i-2})
}
\tag{15.4}
$$

展开：

$$
\boxed{
\bar D
=
\left(1+\frac1{2r_i}\right)D_{i-1}
-
\frac1{2r_i}D_{i-2}
}
\tag{15.5}
$$

这就是 DPM-Solver++(2M) 原论文中的历史预测组合。

### 15.4 最终更新公式

将：

$$
\bar D
$$

代入一阶数据预测形式的指数更新：

$$
x_i
=
\frac{\sigma_i}{\sigma_{i-1}}x_{i-1}
+
\alpha_i(1-e^{-h_i})\bar D.
$$

得到：

$$
\boxed{
\begin{aligned}
x_i
&=
\frac{\sigma_i}{\sigma_{i-1}}x_{i-1}\\
&\quad+
\alpha_i(1-e^{-h_i})
\left[
\left(1+\frac1{2r_i}\right)D_{i-1}
-
\frac1{2r_i}D_{i-2}
\right].
\end{aligned}
}
\tag{15.6}
$$

这对应 DPM-Solver++ 原论文 Algorithm 2 的二阶多步更新。([arxiv.org](https://arxiv.org/pdf/2211.01095))

### 15.5 为什么不需要每步两次推理？

因为在第 $i$ 步：

$$
D_{i-2}
$$

是上一轮保存的网络输出。

当前只需要计算：

$$
D_{i-1}.
$$

因此：

$$
\boxed{
\text{每个新时间节点通常只需一次网络推理}
}
$$

不过第一步没有足够的历史信息。

因此论文先使用一阶更新：

$$
\boxed{
x_1
=
\frac{\sigma_1}{\sigma_0}x_0
+
\alpha_1(1-e^{-h_1})D_0
}
\tag{15.7}
$$

之后才进入二阶多步更新。

### 15.6 为什么 2M 在相同 NFE 下可能比 2S 更好？

假设总预算：

$$
\mathrm{NFE}=20.
$$

对于 2S，每步需要两次网络推理，因此大约能够进行：

$$
10
$$

次积分更新。

对于 2M，每个新节点通常只需要一次网络推理，因此可以使用约：

$$
20
$$

个时间区间。

在相同总积分区间长度下，2M 的单步长度通常更小。

而二阶方法的局部误差为：

$$
O(h^3).
$$

因此，更小的步长可能带来更低的累计离散化误差。

但这不是无条件成立的。

因为多步方法还受到：

- 历史预测误差；
- 步长比例；
- 初始一阶步骤；
- 模型输出平滑性；
- 数值稳定性；

等因素影响。

---

## 16. 为什么原论文 2M 与某些实现公式不完全相同？

前面推导过精确保留指数积分一阶矩的二阶方法。

它的修正项为：

$$
\boxed{
\alpha_i
\frac{h_i-1+e^{-h_i}}{h_{i-1}}
(D_{i-1}-D_{i-2})
}
\tag{16.1}
$$

而 DPM-Solver++ 原论文 Algorithm 2 的修正项为：

$$
\boxed{
\alpha_i
\frac{1-e^{-h_i}}{2r_i}
(D_{i-1}-D_{i-2})
}
\tag{16.2}
$$

因为：

$$
r_i=\frac{h_{i-1}}{h_i},
$$

所以第二个系数也可以写成：

$$
\alpha_i
\frac{h_i(1-e^{-h_i})}{2h_{i-1}}.
$$

现在比较两者。

对指数函数展开：

$$
e^{-h}
=
1-h+\frac{h^2}{2}-\frac{h^3}{6}+O(h^4).
$$

因此：

$$
h-1+e^{-h}
=
\frac{h^2}{2}-\frac{h^3}{6}+O(h^4).
$$

而：

$$
\frac h2(1-e^{-h})
=
\frac{h^2}{2}-\frac{h^3}{4}+O(h^4).
$$

两者的二阶主项相同：

$$
\frac{h^2}{2}.
$$

差异从：

$$
O(h^3)
$$

开始。

因此两种方法都可以具有二阶精度，但高阶误差常数不同。

这解释了为什么不同 DPM-Solver++ 实现中可能出现不同的二阶系数。

**阅读代码时，不能只根据名字 `2M` 判断它使用了哪一个具体变体。**

---

## 17. Dynamic Thresholding：为什么需要限制数据预测？

DPM-Solver++ 还研究强引导条件下的数据范围问题。

假设训练数据归一化为：

$$
x_0\in[-1,1]^D.
$$

在强 CFG 下，数据预测：

$$
D_\theta(x_t,t,c)
$$

可能超出该范围。

如果直接使用：

$$
D_\theta
$$

进行更新，可能导致生成结果过度饱和或出现其他失真。

### 17.1 简单裁剪

最直接的方式是：

$$
\boxed{
\hat D_\theta
=
\operatorname{clip}(D_\theta,-1,1)
}
\tag{17.1}
$$

但简单裁剪可能丢失较多动态范围信息。

### 17.2 Dynamic Thresholding

Dynamic Thresholding 首先计算预测数据绝对值的某个分位数：

$$
s_t
=
\operatorname{Quantile}_p(|D_\theta|).
$$

然后定义：

$$
\boxed{
s_t\leftarrow\max(s_t,1)
}
\tag{17.2}
$$

最后：

$$
\boxed{
\hat D_\theta
=
\frac{
\operatorname{clip}(D_\theta,-s_t,s_t)
}{
s_t
}
}
\tag{17.3}
$$

这样能够根据当前预测的幅度，自适应地进行裁剪和缩放。

不过，这种操作有明确的适用前提。

对于图像：

$$
x_0\in[-1,1]^D
$$

通常具有合理的归一化边界。

但对于自动驾驶轨迹：

$$
x_0\in\mathbb R^{T\times d},
$$

坐标、速度和航向等变量未必共享相同的有界区间。

因此不能直接将图像生成中的 Dynamic Thresholding 原样用于轨迹规划。

此外，论文也讨论了潜空间扩散模型的情况；潜变量通常没有像素空间那样简单的固定边界，因此 thresholding 并不总是合适。([arxiv.org](https://arxiv.org/pdf/2211.01095))

---

## 18. SDE-DPM-Solver++：随机采样扩展

前面讲解的 DPM-Solver 和 DPM-Solver++(2S/2M)，主要是确定性 ODE 求解器。

但 DPM-Solver++ 论文还研究了对应的随机采样方法：

$$
\boxed{
\text{SDE-DPM-Solver++}
}
$$

这是原论文 Section 5 的内容。

### 18.1 ODE 与 SDE 的区别

Probability Flow ODE：

$$
\boxed{
dx_t
=
\left[
f(t)x_t-\frac12g^2(t)s_\theta(x_t,t)
\right]dt
}
\tag{18.1}
$$

反向 SDE：

$$
\boxed{
dx_t
=
\left[
f(t)x_t-g^2(t)s_\theta(x_t,t)
\right]dt
+
g(t)d\bar W_t
}
\tag{18.2}
$$

其中：

$$
s_\theta(x_t,t)
\approx
\nabla_x\log p_t(x_t).
$$

两者在理想 Score 条件下具有相同的单时刻边缘分布，但轨迹不同。

ODE 的轨迹由初始噪声确定。

SDE 在反向过程中持续注入随机性。

### 18.2 为什么不能直接把 ODE 更新公式加上高斯噪声？

因为 SDE 与 ODE 的漂移项不同。

对于 ODE：

$$
-\frac12g^2s_\theta.
$$

对于反向 SDE：

$$
-g^2s_\theta.
$$

因此，如果只是对：

$$
x_{i+1}^{\mathrm{ODE}}
$$

额外加上：

$$
\eta Z,
$$

一般不能保证得到与目标反向 SDE 一致的离散化。

需要重新考虑：

- SDE 漂移项；
- 随机积分；
- 每步噪声方差；
- 状态转移的条件分布。

### 18.3 SDE-DPM-Solver++ 的思想

论文针对 VP 类型扩散过程，在 logSNR 空间重新推导反向 SDE 的积分表达式。

与 ODE 版本类似，仍然尽量解析处理已知的线性结构。

但最终更新还包含一个随机高斯项。

其一般结构可以概括为：

$$
\boxed{
x_t
=
\text{解析线性传播}
+
\text{数据预测积分}
+
\text{高斯随机项}
}
\tag{18.3}
$$

论文进一步构造了一阶和二阶多步随机求解器。

其中一阶 SDE-DPM-Solver++ 可以与特定随机性参数的 DDIM 更新建立等价关系。

需要注意：这里的 SDE 扩展与前面适用于一般单调 SNR 调度的确定性 ODE 推导，适用条件并不完全相同。论文的这部分推导专门考虑了 VP 类型扩散模型。[arXiv](https://arxiv.org/pdf/2211.01095?utm_source=chatgpt.com)

---



### 18.4 SDE-DPM-Solver++ 详解：从反向 SDE 到二阶随机指数积分

**SDE-DPM-Solver++ 是 DPM-Solver++ 论文提出的一种快速随机采样器。它不是在确定性 DPM-Solver++ 的结果上随意添加高斯噪声，而是直接从扩散模型的反向 SDE 出发，推导精确积分表达式，再对其中的神经网络预测项进行一阶或二阶近似。**

它保留了 DPM-Solver++ 的两个关键设计：

1. 使用数据预测模型 $D_\theta(x,t)$，而不是直接对噪声预测进行高阶展开。
2. 利用扩散方程的线性结构，解析计算线性传播和随机积分，只近似神经网络相关的非线性积分。

与确定性 DPM-Solver++ 相比，最大的区别是每一步还包含一个**方差可以解析计算的高斯随机项**。

这部分对应论文 *DPM-Solver++: Fast Solver for Guided Sampling of Diffusion Probabilistic Models* 的 **Section 5、Proposition 5.1、式（10）—（20）**，以及附录 A 的推导。[arXiv](https://arxiv.org/pdf/2211.01095)

---

#### 一、先明确：它到底在求解什么？

##### 1.1 扩散模型的两个反向生成过程

我们首先定义正向扩散过程：

$$
\boxed{
x_t=\alpha_t x_0+\sigma_t\epsilon
}
\tag{1.1}
$$

其中：

- $x_0\sim p_{\mathrm{data}}$：真实数据；
- $\epsilon\sim\mathcal N(0,I)$：标准高斯噪声；
- $\alpha_t$：信号系数；
- $\sigma_t$：噪声标准差；
- $t\in[0,T]$：正向扩散时间。

生成时从 $T$ 反向走到 $0$。

对于扩散模型，我们有两种常见的生成动力学。

**第一种：Probability Flow ODE**

$$
\boxed{
dx_t=
\left[
f(t)x_t
-\frac12g^2(t)\nabla_x\log p_t(x_t)
\right]dt
}
\tag{1.2}
$$

**第二种：反向 SDE**

$$
\boxed{
dx_t=
\left[
f(t)x_t
-g^2(t)\nabla_x\log p_t(x_t)
\right]dt
+
g(t)d\bar W_t
}
\tag{1.3}
$$

其中：

- $f(t)$：正向 SDE 的线性漂移系数；
- $g(t)$：正向 SDE 的扩散系数；
- $\nabla_x\log p_t(x)$：边缘分布的 Score；
- $\bar W_t$：按反向时间理解的布朗运动。

式（1.2）没有随机项，是确定性 ODE。

式（1.3）包含随机项，是反向 SDE。

在理想 Score 和适当正则性条件下，二者具有相同的单时刻边缘分布：

$$
\boxed{
p_t^{\mathrm{ODE}}(x)=p_t^{\mathrm{SDE}}(x)
}
\tag{1.4}
$$

但这不代表它们具有相同的轨迹或相同的条件转移分布。

##### 1.2 使用噪声预测模型

根据：

$$
\boxed{
\nabla_x\log p_t(x)
\approx
-\frac{\epsilon_\theta(x,t)}{\sigma_t}
}
\tag{1.5}
$$

反向 SDE 变成：

$$
\boxed{
dx_t=
\left[
f(t)x_t
+
\frac{g^2(t)}{\sigma_t}
\epsilon_\theta(x_t,t)
\right]dt
+
g(t)d\bar W_t
}
\tag{1.6}
$$

对比 PF ODE：

$$
\boxed{
dx_t=
\left[
f(t)x_t
+
\frac{g^2(t)}{2\sigma_t}
\epsilon_\theta(x_t,t)
\right]dt
}
\tag{1.7}
$$

可以发现：

**反向 SDE 中的 Score 修正项，是 PF ODE 中对应修正项的两倍。**

此外，反向 SDE 还包含：

$$
g(t)d\bar W_t.
$$

所以：

$$
\boxed{
\text{反向 SDE}
\neq
\text{PF ODE}+\text{任意高斯噪声}
}
\tag{1.8}
$$

这是理解 SDE-DPM-Solver++ 的第一个关键点。

##### 1.3 为什么还要专门设计 SDE 求解器？

如果直接使用 Euler–Maruyama，可以写成概念上的反向更新：

$$
x_{t-\Delta t}
\approx
x_t
-
\Delta t\,b_\theta(x_t,t)
+
g(t)\sqrt{\Delta t}\,z,
$$

其中：

$$
z\sim\mathcal N(0,I),
$$

$$
b_\theta(x,t)
=
f(t)x+\frac{g^2(t)}{\sigma_t}\epsilon_\theta(x,t).
$$

问题在于：Euler–Maruyama 对漂移项和随机扩散项都采用局部离散近似。

而扩散模型的 SDE 具有特殊结构：

$$
\boxed{
dx_t=
\underbrace{f(t)x_tdt}_{\text{已知线性项}}
+
\underbrace{\frac{g^2(t)}{\sigma_t}\epsilon_\theta dt}_{\text{网络预测项}}
+
\underbrace{g(t)d\bar W_t}_{\text{随机项}}
}
\tag{1.9}
$$

其中：

- 线性项可以解析积分；
- 随机积分的分布可以解析计算；
- 只有网络预测项需要近似。

因此作者希望构造：

$$
\boxed{
\text{精确线性传播}
+
\text{高阶网络积分}
+
\text{精确方差的随机项}
}
\tag{1.10}
$$

这就是 SDE-DPM-Solver++ 的核心思想。

---

#### 二、为什么论文首先限定 VP 扩散模型？

论文 Section 5 的推导明确考虑 VP-type diffusion：

$$
\boxed{
\alpha_t^2+\sigma_t^2=1
}
\tag{2.1}
$$

这是与前面确定性 DPM-Solver++ 需要区分的地方。

原始 DPM-Solver++ 的 ODE 精确积分表达式适用于更一般的单调信噪比调度；而论文这里给出的 SDE 推导利用了 VP 关系。

定义半对数信噪比：

$$
\boxed{
\lambda_t=\log\frac{\alpha_t}{\sigma_t}
}
\tag{2.2}
$$

##### 记号说明：从原始时间重新参数化

严格地说，$\alpha_t=\alpha(t)$、$\sigma_t=\sigma(t)$，而 $\lambda(t)=\log[\alpha(t)/\sigma(t)]$ 也是 $t$ 的函数。在所讨论的区间内，假设 $\lambda(t)$ 严格递减并可逆，记其反函数为 $t=\tau(\ell)$。定义

$$
\widetilde\alpha(\ell)=\alpha(\tau(\ell)),\qquad
\widetilde\sigma(\ell)=\sigma(\tau(\ell)).
$$

下文的 $\alpha_\lambda$、$\sigma_\lambda$ 就是 $\widetilde\alpha(\lambda)$、$\widetilde\sigma(\lambda)$ 的简写。这里是对函数的自变量重新参数化，不是直接把下标 $t$ 改成 $\lambda$。由 VP 条件和 $e^\ell=\widetilde\alpha(\ell)/\widetilde\sigma(\ell)$，才能得到下面的两个平方公式；取正根还需 $\alpha,\sigma>0$。

因此：

$$
e^{\lambda_t}=\frac{\alpha_t}{\sigma_t}.
$$

由于：

$$
\alpha_t^2+\sigma_t^2=1,
$$

可以得到：

$$
\boxed{
\alpha_\lambda^2
=
\frac{e^{2\lambda}}{1+e^{2\lambda}}
}
\tag{2.3}
$$

以及：

$$
\boxed{
\sigma_\lambda^2
=
\frac1{1+e^{2\lambda}}
}
\tag{2.4}
$$

下面推导两个非常重要的导数关系。

##### 2.1 推导 $d\log\alpha/d\lambda$

由：

$$
\log\alpha_\lambda
=
\lambda-\frac12\log(1+e^{2\lambda}),
$$

求导：

$$
\begin{aligned}
\frac{d\log\alpha_\lambda}{d\lambda}
&=
1-\frac{e^{2\lambda}}{1+e^{2\lambda}}\\
&=
\frac1{1+e^{2\lambda}}.
\end{aligned}
$$

因此：

$$
\boxed{
\frac{d\log\alpha_\lambda}{d\lambda}
=
\sigma_\lambda^2
}
\tag{2.5}
$$

##### 2.2 推导 $d\log\sigma/d\lambda$

由：

$$
\log\sigma_\lambda
=
-\frac12\log(1+e^{2\lambda}),
$$

求导：

$$
\begin{aligned}
\frac{d\log\sigma_\lambda}{d\lambda}
&=
-\frac{e^{2\lambda}}{1+e^{2\lambda}}\\
&=
-\alpha_\lambda^2.
\end{aligned}
$$

因此：

$$
\boxed{
\frac{d\log\sigma_\lambda}{d\lambda}
=
-\alpha_\lambda^2
}
\tag{2.6}
$$

这两个关系后面会直接用于推导 SDE 的精确积分公式。

---

#### 三、从反向时间 $t$ 转换到 logSNR 时间 $\lambda$

##### 3.1 首先处理漂移项

反向 SDE：

$$
dx_t=
\left[
f(t)x_t+
\frac{g^2(t)}{\sigma_t}
\epsilon_\theta(x_t,t)
\right]dt
+
g(t)d\bar W_t.
$$

根据前面 DPM-Solver 的数学基础：

$$
\boxed{
f(t)=\frac{d\log\alpha_t}{dt}
}
\tag{3.1}
$$

以及：

$$
\boxed{
g^2(t)=-2\sigma_t^2\dot\lambda_t
}
\tag{3.2}
$$

注意正向时间中：

$$
\dot\lambda_t<0.
$$

因为正向扩散时信噪比不断降低。

现在将时间变量从 $t$ 改为 $\lambda$。

对漂移项使用：

$$
dt=\frac{d\lambda}{\dot\lambda_t}.
$$

第一项：

$$
\begin{aligned}
f(t)x_tdt
&=
\frac{d\log\alpha_t}{dt}
x_t
\frac{d\lambda}{\dot\lambda_t}\\
&=
\frac{d\log\alpha_\lambda}{d\lambda}
x_\lambda d\lambda.
\end{aligned}
$$

根据式（2.5）：

$$
\boxed{
f(t)x_tdt
=
\sigma_\lambda^2x_\lambda d\lambda
}
\tag{3.3}
$$

第二项：

$$
\begin{aligned}
\frac{g^2(t)}{\sigma_t}\epsilon_\theta dt
&=
\frac{-2\sigma_t^2\dot\lambda_t}{\sigma_t}
\epsilon_\theta
\frac{d\lambda}{\dot\lambda_t}\\
&=
-2\sigma_\lambda\epsilon_\theta d\lambda.
\end{aligned}
$$

因此：

$$
\boxed{
\frac{g^2(t)}{\sigma_t}\epsilon_\theta dt
=
-2\sigma_\lambda\epsilon_\theta d\lambda
}
\tag{3.4}
$$

##### 3.2 再处理随机项：布朗运动的时间变换

随机微分不能按普通微积分把 $dt=d\lambda/\dot\lambda_t$ 直接代入 $d\bar W_t$。标准布朗运动增量在长度为 $\Delta r$ 的时间区间上方差为 $\Delta r$，标准差为 $\sqrt{\Delta r}$。例如若 $\ell=2r$，则 $B_{\ell/2}$ 的增量方差为 $\Delta\ell/2$；$W_\ell=\sqrt{2}B_{\ell/2}$ 才是以 $\ell$ 为时间参数的标准布朗运动。

为明确生成方向，令反向时间 $r=T-t$，并用 $B_r$ 表示在递增的 $r$ 下驱动反向 SDE 的标准布朗运动。再定义

$$
\Lambda(r)=\lambda(T-r),\qquad
\frac{d\Lambda}{dr}=-\dot\lambda_{T-r}>0.
$$

于是 $dr=d\Lambda/(-\dot\lambda_t)$。根据布朗运动的确定性时间变换，在随机微分意义下可以引入相对于递增 $\Lambda$ 的标准布朗运动 $W_\Lambda$，使得

$$
dB_r=\frac{1}{\sqrt{-\dot\lambda_t}}\,dW_\Lambda,
\qquad t=T-r.
$$

这个等式表达的是随机积分的时间变换，不是逐路径可微的链式法则。平方根来自方差的匹配：左边的方差为 $dr$，右边的方差为 $d\Lambda/(-\dot\lambda_t)=dr$。$W_\Lambda$ 是新时间坐标下的标准布朗运动，不能仅把原来布朗运动的下标改名。

VP 扩散满足 $g^2(t)=-2\sigma_t^2\dot\lambda_t$。取非负扩散系数 $g(t)$ 后，随机项变为

$$
g(t)dB_r
=\frac{g(t)}{\sqrt{-\dot\lambda_t}}\,dW_\Lambda
=\sqrt{2}\,\sigma_t\,dW_\Lambda
=\sqrt{2}\,\sigma_\Lambda\,dW_\Lambda.
\tag{3.5}
$$

这就是下式中随机系数 $\sqrt{2}\sigma_\lambda$ 的来源。漂移项仍按普通时间变换处理；原始反向 SDE 中沿递减 $t$ 的 $dt$ 号与递增 $r$ 的 $dr$ 号应一并转换。

##### 3.3 得到 logSNR 空间的反向 SDE

将式（3.3）—（3.5）组合：

$$
\boxed{
dx_\lambda
=
\left[
\sigma_\lambda^2x_\lambda
-
2\sigma_\lambda
\epsilon_\theta(x_\lambda,\lambda)
\right]d\lambda
+
\sqrt2\sigma_\lambda dW_\lambda
}
\tag{3.6}
$$

这就是论文式（11）。

注意时间方向：

$$
\boxed{
\lambda:\lambda_T\rightarrow\lambda_0
}
$$

是递增的。

这里的两个方向不要混淆：原始扩散时间从 $T$ 走向 $0$，反向时间 $r$ 从 $0$ 递增，$\lambda$ 从较小的 $\lambda_T$ 递增到较大的 $\lambda_0$。例如 $(\alpha_T,\sigma_T)\approx(0.01,1)$ 与 $(\alpha_\epsilon,\sigma_\epsilon)\approx(1,0.01)$ 分别对应 $\lambda_T\approx-4.605$ 和 $\lambda_\epsilon\approx4.605$。$\lambda=0$ 仅表示 $\alpha=\sigma$（SNR 为 1），并非生成终点。论文用 $s>t$ 表示从高噪声采样到低噪声，因此 $\lambda_s<\lambda_t$，积分 $\int_{\lambda_s}^{\lambda_t}$ 按递增顺序计算。

因此后面的随机积分都可以按照正常的递增时间处理。

---

#### 四、将反向 SDE 转换为数据预测形式

定义：

$$
\boxed{
D_\theta(x_\lambda,\lambda)
=
\frac{x_\lambda-\sigma_\lambda\epsilon_\theta(x_\lambda,\lambda)}
{\alpha_\lambda}
}
\tag{4.1}
$$

反过来：

$$
\boxed{
\epsilon_\theta
=
\frac{x_\lambda-\alpha_\lambda D_\theta}{\sigma_\lambda}
}
\tag{4.2}
$$

代入式（3.6）：

$$
dx_\lambda
=
\left[
\sigma_\lambda^2x_\lambda
-
2\sigma_\lambda
\frac{x_\lambda-\alpha_\lambda D_\theta}{\sigma_\lambda}
\right]d\lambda
+
\sqrt2\sigma_\lambda dW_\lambda.
$$

整理漂移项：

$$
\sigma_\lambda^2x_\lambda
-
2x_\lambda
+
2\alpha_\lambda D_\theta.
$$

由于：

$$
\sigma_\lambda^2=1-\alpha_\lambda^2,
$$

所以：

$$
\sigma_\lambda^2-2
=
-(1+\alpha_\lambda^2).
$$

最终：

$$
\boxed{
dx_\lambda
=
\left[
-(1+\alpha_\lambda^2)x_\lambda
+
2\alpha_\lambda D_\theta(x_\lambda,\lambda)
\right]d\lambda
+
\sqrt2\sigma_\lambda dW_\lambda
}
\tag{4.3}
$$

这就是论文式（12）。

现在我们得到一个非常清楚的结构：

$$
\boxed{
dx_\lambda
=
\underbrace{-(1+\alpha_\lambda^2)x_\lambda d\lambda}_{\text{已知线性项}}
+
\underbrace{2\alpha_\lambda D_\theta d\lambda}_{\text{网络预测项}}
+
\underbrace{\sqrt2\sigma_\lambda dW_\lambda}_{\text{随机项}}
}
\tag{4.4}
$$

接下来要做的，就是分别处理这三项。

---

#### 五、核心推导：SDE-DPM-Solver++ 的精确积分公式

这一节是整个算法最重要的部分，对应论文 **Proposition 5.1**。

##### 5.1 首先求解线性部分

从式（4.3）出发：

$$
dx_\lambda
=
a(\lambda)x_\lambda d\lambda
+
2\alpha_\lambda D_\theta d\lambda
+
\sqrt2\sigma_\lambda dW_\lambda,
$$

其中：

$$
\boxed{
a(\lambda)=-(1+\alpha_\lambda^2)
}
\tag{5.1}
$$

对于一般线性非齐次 SDE：

$$
dx_\lambda
=
a(\lambda)x_\lambda d\lambda
+
b_\lambda d\lambda
+
c_\lambda dW_\lambda,
$$

定义线性传播因子：

$$
\boxed{
\Phi(\lambda,u)
=
\exp\left(
\int_u^\lambda a(v)\,dv
\right)
}
\tag{5.2}
$$

则变常数公式给出：

$$
\boxed{
\begin{aligned}
x_{\lambda_t}
={}&
\Phi(\lambda_t,\lambda_s)x_{\lambda_s}\\
&+
\int_{\lambda_s}^{\lambda_t}
\Phi(\lambda_t,u)b_u\,du\\
&+
\int_{\lambda_s}^{\lambda_t}
\Phi(\lambda_t,u)c_u\,dW_u.
\end{aligned}
}
\tag{5.3}
$$

因此首先需要计算：

$$
\Phi(\lambda_t,u).
$$

##### 5.2 解析计算传播因子

根据式（2.6）：

$$
\frac{d\log\sigma_\lambda}{d\lambda}
=
-\alpha_\lambda^2.
$$

因此：

$$
\begin{aligned}
a(\lambda)
&=
-(1+\alpha_\lambda^2)\\
&=
-1+\frac{d\log\sigma_\lambda}{d\lambda}.
\end{aligned}
$$

于是：

$$
\begin{aligned}
\int_u^{\lambda_t}a(v)\,dv
&=
-\int_u^{\lambda_t}1\,dv
+
\int_u^{\lambda_t}d\log\sigma_v\\
&=
-(\lambda_t-u)
+
\log\frac{\sigma_t}{\sigma_u}.
\end{aligned}
$$

取指数：

$$
\boxed{
\Phi(\lambda_t,u)
=
\frac{\sigma_t}{\sigma_u}
e^{-(\lambda_t-u)}
}
\tag{5.4}
$$

特别地，定义：

$$
\boxed{
h=\lambda_t-\lambda_s>0
}
\tag{5.5}
$$

则：

$$
\boxed{
\Phi(\lambda_t,\lambda_s)
=
\frac{\sigma_t}{\sigma_s}e^{-h}
}
\tag{5.6}
$$

这就是后面更新公式中：

$$
\frac{\sigma_t}{\sigma_s}e^{-h}x_s
$$

的来源。

注意它与确定性 DPM-Solver++ 中的线性传播系数：

$$
\frac{\sigma_t}{\sigma_s}
$$

不同。

这是因为反向 SDE 与 PF ODE 的漂移不同。

##### 5.3 代入神经网络积分

根据式（5.3），网络预测积分为：

$$
I_D
=
\int_{\lambda_s}^{\lambda_t}
\Phi(\lambda_t,u)
2\alpha_uD_\theta(x_u,u)\,du.
$$

代入式（5.4）：

$$
I_D
=
\int_{\lambda_s}^{\lambda_t}
\frac{\sigma_t}{\sigma_u}
e^{-(\lambda_t-u)}
2\alpha_uD_\theta(x_u,u)\,du.
$$

利用：

$$
\frac{\alpha_u}{\sigma_u}=e^u,
$$

得到：

$$
\begin{aligned}
I_D
&=
2\sigma_t
\int_{\lambda_s}^{\lambda_t}
e^{-(\lambda_t-u)}
e^uD_\theta(x_u,u)\,du\\
&=
2\sigma_t e^{-\lambda_t}
\int_{\lambda_s}^{\lambda_t}
e^{2u}D_\theta(x_u,u)\,du.
\end{aligned}
$$

又因为：

$$
\sigma_t e^{\lambda_t}=\alpha_t,
$$

所以：

$$
\sigma_t e^{-\lambda_t}
=
\alpha_t e^{-2\lambda_t}.
$$

因此：

$$
\boxed{
I_D
=
2\alpha_t
\int_{\lambda_s}^{\lambda_t}
e^{-2(\lambda_t-u)}
D_\theta(x_u,u)\,du
}
\tag{5.7}
$$

这就是 SDE-DPM-Solver++ 的网络积分部分。

对比确定性版本：

$$
\boxed{
I_D^{\mathrm{ODE}}
=
\sigma_t
\int_{\lambda_s}^{\lambda_t}
e^uD_\theta(x_u,u)\,du
}
\tag{5.8}
$$

可以看到，SDE 和 ODE 不只是相差一个随机项，它们对网络预测的积分权重也不同。

##### 5.4 推导随机积分

随机部分：

$$
I_W
=
\sqrt2
\int_{\lambda_s}^{\lambda_t}
\Phi(\lambda_t,u)\sigma_u\,dW_u.
$$

代入传播因子：

$$
\begin{aligned}
I_W
&=
\sqrt2
\int_{\lambda_s}^{\lambda_t}
\frac{\sigma_t}{\sigma_u}
e^{-(\lambda_t-u)}
\sigma_u\,dW_u\\
&=
\sqrt2\sigma_t
\int_{\lambda_s}^{\lambda_t}
e^{-(\lambda_t-u)}dW_u.
\end{aligned}
$$

因此：

$$
\boxed{
I_W
=
\sqrt2\sigma_t
\int_{\lambda_s}^{\lambda_t}
e^{-(\lambda_t-u)}dW_u
}
\tag{5.9}
$$

这个随机积分可以精确确定其概率分布。

因为被积函数：

$$
e^{-(\lambda_t-u)}
$$

是确定性函数，所以 Itô 积分服从零均值高斯分布。

更完整的理由是：若 $f\in L^2([a,b])$ 是确定性函数，则 $\int_a^b f(u)\,dW_u$ 是零均值高斯变量。对分段常数 $f$，该积分是互相独立的高斯增量 $\Delta W_i$ 的确定性线性组合；一般平方可积的 $f$ 可由分段常数函数在 $L^2$ 中逼近。因此

$$
\int_a^b f(u)\,dW_u
\sim\mathcal N\!\left(0,\int_a^b f^2(u)\,du\right).
$$

Itô 等距公式给出的是二阶矩，并不能单独推出高斯性；一般随机被积函数的积分也未必是高斯变量。例如 $\int_0^T W_u\,dW_u=(W_T^2-T)/2$，它是中心化的 $\chi^2$ 型变量。本步中 $f(u)=\sqrt{2}\sigma_t e^{-(\lambda_t-u)}$ 是确定性函数，因此可先确定高斯性，再用等距公式计算方差。

根据 Itô 等距公式：

$$
\boxed{
\mathbb E[I_W^2]
=
2\sigma_t^2
\int_{\lambda_s}^{\lambda_t}
e^{-2(\lambda_t-u)}du
}
\tag{5.10}
$$

令：

$$
v=\lambda_t-u,
$$

则：

$$
u=\lambda_s\Rightarrow v=h,
$$

$$
u=\lambda_t\Rightarrow v=0.
$$

因此：

$$
\begin{aligned}
\mathbb E[I_W^2]
&=
2\sigma_t^2
\int_0^h e^{-2v}dv\\
&=
2\sigma_t^2
\left[
-\frac12e^{-2v}
\right]_0^h\\
&=
\sigma_t^2(1-e^{-2h}).
\end{aligned}
$$

所以：

$$
\boxed{
I_W
\sim
\mathcal N
\left(
0,\,
\sigma_t^2(1-e^{-2h})I
\right)
}
\tag{5.11}
$$

等价地：

$$
\boxed{
I_W
\overset{d}{=}
\sigma_t\sqrt{1-e^{-2h}}\,z,
\qquad
z\sim\mathcal N(0,I)
}
\tag{5.12}
$$

其中 $\overset{d}{=}$ 表示分布相同。

**这是 SDE-DPM-Solver++ 的第二个关键：随机项不需要用 Euler–Maruyama 的 $g(t)\sqrt{\Delta t}$ 近似，而是能够直接计算该区间内随机积分的精确方差。**

但这里的“精确”只针对这个具有确定性积分核的随机项，并不意味着整个非线性 SDE 的一步转移已经精确求出。

##### 5.5 最终精确积分表达式

将三部分组合：

$$
\boxed{
\begin{aligned}
x_t
={}&
\frac{\sigma_t}{\sigma_s}e^{-h}x_s\\
&+
2\alpha_t
\int_{\lambda_s}^{\lambda_t}
e^{-2(\lambda_t-\lambda)}
D_\theta(x_\lambda,\lambda)\,d\lambda\\
&+
\sqrt2\sigma_t
\int_{\lambda_s}^{\lambda_t}
e^{-(\lambda_t-\lambda)}dW_\lambda.
\end{aligned}
}
\tag{5.13}
$$

这就是论文 Proposition 5.1 中的**数据预测形式的精确 SDE 积分表达式**。

注意最后一项是随机积分，而不是普通积分。

虽然它的边缘分布可以用式（5.12）中的高斯变量表示，但中间轨迹 $x_\lambda$ 本身也受布朗运动影响。

因此，网络积分：

$$
\int_{\lambda_s}^{\lambda_t}
e^{-2(\lambda_t-\lambda)}
D_\theta(x_\lambda,\lambda)d\lambda
$$

仍然是沿随机轨迹计算的非线性积分。

还要区分边缘分布与联合分布：式（5.12）只给出随机积分 $I_W$ 自身的分布。精确 SDE 的网络积分沿同一布朗运动驱动的随机轨迹计算，两项一般相关。把 $I_W$ 表示为新采样的高斯变量，是在对网络积分作相应离散近似后实现求解器一步更新的方式。

这也是接下来必须进行数值近似的原因。[arXiv](https://arxiv.org/pdf/2211.01095)

---

#### 六、SDE-DPM-Solver++1：一阶随机采样器

##### 6.1 对网络预测使用常数近似

从精确积分式（5.13）出发：

$$
\begin{aligned}
x_t
={}&
\frac{\sigma_t}{\sigma_s}e^{-h}x_s\\
&+
2\alpha_t
\int_{\lambda_s}^{\lambda_t}
e^{-2(\lambda_t-\lambda)}
D_\theta(x_\lambda,\lambda)d\lambda\\
&+
I_W.
\end{aligned}
$$

一阶方法假设：

$$
\boxed{
D_\theta(x_\lambda,\lambda)
\approx
D_\theta(x_s,s)
}
\tag{6.1}
$$

记：

$$
D_s=D_\theta(x_s,s).
$$

则：

$$
\begin{aligned}
I_D
&\approx
2\alpha_tD_s
\int_{\lambda_s}^{\lambda_t}
e^{-2(\lambda_t-\lambda)}d\lambda.
\end{aligned}
$$

根据前面的积分计算：

$$
\int_{\lambda_s}^{\lambda_t}
e^{-2(\lambda_t-\lambda)}d\lambda
=
\frac{1-e^{-2h}}2.
$$

因此：

$$
\boxed{
I_D
\approx
\alpha_t(1-e^{-2h})D_s
}
\tag{6.2}
$$

随机积分使用式（5.12）。

最终得到：

$$
\boxed{
\begin{aligned}
x_t
={}&
\frac{\sigma_t}{\sigma_s}e^{-h}x_s\\
&+
\alpha_t(1-e^{-2h})D_\theta(x_s,s)\\
&+
\sigma_t\sqrt{1-e^{-2h}}\,z,
\end{aligned}
}
\tag{6.3}
$$

其中：

$$
z\sim\mathcal N(0,I).
$$

这就是论文式（18）：

$$
\boxed{
\text{SDE-DPM-Solver++1}
}
$$

##### 6.2 从条件高斯分布理解

固定当前状态 $x_s$，并将网络预测：

$$
D_s=D_\theta(x_s,s)
$$

视为已知。

那么式（6.3）具有：

$$
\boxed{
x_t\mid x_s
\sim
\mathcal N
\left(
\mu_\theta(x_s;s,t),
\Sigma_{s,t}
\right)
}
\tag{6.4}
$$

其中：

$$
\boxed{
\mu_\theta
=
\frac{\sigma_t}{\sigma_s}e^{-h}x_s
+
\alpha_t(1-e^{-2h})D_s
}
\tag{6.5}
$$

以及：

$$
\boxed{
\Sigma_{s,t}
=
\sigma_t^2(1-e^{-2h})I
}
\tag{6.6}
$$

因此，一阶 SDE-DPM-Solver++ 可以直接理解为：

> 每一步根据当前状态和网络预测计算高斯转移的均值，再使用解析得到的方差采样下一个状态。

但是必须强调：

**式（6.4）是离散化求解器定义的一步条件转移，不是一般非线性反向 SDE 的精确条件转移。**

因为我们已经对：

$$
D_\theta(x_\lambda,\lambda)
$$

使用了常数近似。

##### 6.3 为什么噪声标准差是这个形式？

随机项：

$$
\sigma_t\sqrt{1-e^{-2h}}\,z
$$

可以分为两个因素：

$$
\underbrace{\sigma_t}_{\text{目标时间的噪声尺度}}
$$

和：

$$
\underbrace{\sqrt{1-e^{-2h}}}_{\text{积分区间产生的随机性}}
$$

当：

$$
h\rightarrow0
$$

时：

$$
e^{-2h}
=
1-2h+O(h^2),
$$

所以：

$$
1-e^{-2h}
=
2h+O(h^2).
$$

因此：

$$
\boxed{
\sigma_t\sqrt{1-e^{-2h}}
\approx
\sigma_t\sqrt{2h}
}
\tag{6.7}
$$

这与 logSNR 空间 SDE 的瞬时扩散系数：

$$
\sqrt2\sigma_\lambda
$$

是一致的。

当：

$$
h
$$

较大时，解析表达式又避免了简单使用局部：

$$
\sqrt{2h}\sigma_s
$$

所产生的部分离散化误差。

---

#### 七、SDE-DPM-Solver++1 与随机 DDIM 的关系

这是论文 Section 6.1 给出的一个重要结论。

##### 7.1 一般随机 DDIM 更新

对于从 $s$ 到 $t$ 的采样，随机 DDIM 可以写成：

$$
\boxed{
x_t
=
\alpha_tD_s
+
\sqrt{\sigma_t^2-\eta^2}\,
\epsilon_s
+
\eta z
}
\tag{7.1}
$$

其中：

$$
\epsilon_s=\epsilon_\theta(x_s,s),
$$

$$
D_s=
\frac{x_s-\sigma_s\epsilon_s}{\alpha_s},
$$

$$
z\sim\mathcal N(0,I).
$$

这里的 $\eta$ 表示**当前一步实际注入的噪声标准差**，而不是必须等于某个全局无量纲 DDIM 随机性超参数。

##### 7.2 选择特殊噪声尺度

令：

$$
\boxed{
\eta
=
\sigma_t\sqrt{1-e^{-2h}}
}
\tag{7.2}
$$

那么：

$$
\begin{aligned}
\sqrt{\sigma_t^2-\eta^2}
&=
\sqrt{
\sigma_t^2-
\sigma_t^2(1-e^{-2h})
}\\
&=
\sigma_t e^{-h}.
\end{aligned}
$$

因此随机 DDIM 变成：

$$
\boxed{
x_t
=
\alpha_tD_s
+
\sigma_t e^{-h}\epsilon_s
+
\sigma_t\sqrt{1-e^{-2h}}z
}
\tag{7.3}
$$

##### 7.3 与 SDE-DPM-Solver++1 对比

SDE-DPM-Solver++1：

$$
x_t
=
\frac{\sigma_t}{\sigma_s}e^{-h}x_s
+
\alpha_t(1-e^{-2h})D_s
+
\sigma_t\sqrt{1-e^{-2h}}z.
$$

根据：

$$
x_s=\alpha_sD_s+\sigma_s\epsilon_s,
$$

第一项可以展开：

$$
\begin{aligned}
\frac{\sigma_t}{\sigma_s}e^{-h}x_s
&=
\frac{\sigma_t\alpha_s}{\sigma_s}e^{-h}D_s
+
\sigma_t e^{-h}\epsilon_s.
\end{aligned}
$$

又因为：

$$
e^{-h}
=
\frac{\alpha_s\sigma_t}{\alpha_t\sigma_s},
$$

所以：

$$
\frac{\sigma_t\alpha_s}{\sigma_s}e^{-h}
=
\alpha_t e^{-2h}.
$$

因此：

$$
\begin{aligned}
x_t
={}&
\alpha_t e^{-2h}D_s
+
\sigma_t e^{-h}\epsilon_s\\
&+
\alpha_t(1-e^{-2h})D_s\\
&+
\sigma_t\sqrt{1-e^{-2h}}z.
\end{aligned}
$$

合并数据预测项：

$$
\alpha_t e^{-2h}D_s
+
\alpha_t(1-e^{-2h})D_s
=
\alpha_tD_s.
$$

最终：

$$
\boxed{
x_t
=
\alpha_tD_s
+
\sigma_t e^{-h}\epsilon_s
+
\sigma_t\sqrt{1-e^{-2h}}z
}
\tag{7.4}
$$

这与式（7.3）完全一致。

因此：

$$
\boxed{
\text{SDE-DPM-Solver++1}
=
\text{采用特定逐步噪声尺度的随机 DDIM}
}
\tag{7.5}
$$

这是论文明确给出的等价关系。[arXiv](https://arxiv.org/pdf/2211.01095)

##### 7.4 这个等价关系说明什么？

我们之前已经知道：

$$
\boxed{
\text{DPM-Solver++1（ODE）}
=
\text{DDIM}(\eta=0)
}
$$

现在进一步得到：

$$
\boxed{
\text{SDE-DPM-Solver++1}
=
\text{特定随机性参数的 DDIM}
}
$$

因此可以把 SDE-DPM-Solver++(2M) 理解为：

**沿着随机 DDIM 的 SDE 积分结构，将一阶网络近似推广到二阶多步近似。**

---

#### 八、SDE-DPM-Solver++(2M)：二阶随机多步求解器

前面的一阶方法使用：

$$
D_\theta(x_\lambda,\lambda)\approx D_s.
$$

但真实情况下，网络预测沿随机轨迹不断变化。

SDE-DPM-Solver++(2M) 的目标就是：**利用历史网络输出估计当前区间内的数据预测变化，同时继续使用解析形式计算随机积分。**

这是论文式（20）的核心。

##### 8.1 时间节点与符号定义

考虑连续三个反向采样节点：

$$
r>s>t.
$$

由于 $\lambda$ 在反向采样中递增：

$$
\lambda_r<\lambda_s<\lambda_t.
$$

定义当前步长：

$$
\boxed{
h=\lambda_t-\lambda_s>0
}
\tag{8.1}
$$

历史步长：

$$
\boxed{
h_{\mathrm{prev}}=\lambda_s-\lambda_r>0
}
\tag{8.2}
$$

以及步长比例：

$$
\boxed{
q=\frac{h_{\mathrm{prev}}}{h}
}
\tag{8.3}
$$

记当前网络输出：

$$
\boxed{
D_s=D_\theta(x_s,s)
}
\tag{8.4}
$$

历史网络输出：

$$
\boxed{
D_r=D_\theta(x_r,r)
}
\tag{8.5}
$$

这里特意使用 $q$ 表示步长比例，避免与历史时间 $r$ 混淆。

##### 8.2 使用历史输出估计导数

沿随机轨迹定义：

$$
D(\lambda)=D_\theta(x_\lambda,\lambda).
$$

在当前时间 $\lambda_s$ 附近，形式上使用一阶近似：

$$
D(\lambda)
\approx
D_s+(\lambda-\lambda_s)D_s'.
$$

使用历史差分：

$$
\boxed{
D_s'
\approx
\frac{D_s-D_r}{\lambda_s-\lambda_r}
}
\tag{8.6}
$$

因为：

$$
\lambda_s-\lambda_r=qh,
$$

所以：

$$
\boxed{
D_s'
\approx
\frac{D_s-D_r}{qh}
}
\tag{8.7}
$$

因此：

$$
\boxed{
D(\lambda)
\approx
D_s+
\frac{\lambda-\lambda_s}{qh}(D_s-D_r)
}
\tag{8.8}
$$

这里的差分方向很重要。

因为：

$$
D_s-D_r
$$

表示从历史节点到当前节点的变化。

如果写成：

$$
D_r-D_s,
$$

就必须相应改变分母的符号。

##### 8.3 将线性近似代入精确积分

网络积分为：

$$
I_D
=
2\alpha_t
\int_{\lambda_s}^{\lambda_t}
e^{-2(\lambda_t-\lambda)}
D(\lambda)d\lambda.
$$

代入式（8.8）：

$$
\begin{aligned}
I_D
\approx{}&
2\alpha_tD_s
\int_{\lambda_s}^{\lambda_t}
e^{-2(\lambda_t-\lambda)}d\lambda\\
&+
2\alpha_t
\frac{D_s-D_r}{qh}
\int_{\lambda_s}^{\lambda_t}
e^{-2(\lambda_t-\lambda)}
(\lambda-\lambda_s)d\lambda.
\end{aligned}
\tag{8.9}
$$

将两项分别记为：

$$
I_0,\qquad I_1.
$$

第一项已经计算过：

$$
\boxed{
I_0
=
\alpha_t(1-e^{-2h})D_s
}
\tag{8.10}
$$

现在重点计算第二项。

##### 8.4 计算二阶修正积分

定义：

$$
J=
\int_{\lambda_s}^{\lambda_t}
e^{-2(\lambda_t-\lambda)}
(\lambda-\lambda_s)d\lambda.
$$

令：

$$
u=\lambda-\lambda_s.
$$

于是：

$$
\lambda_t-\lambda=h-u.
$$

所以：

$$
\begin{aligned}
J
&=
\int_0^h e^{-2(h-u)}u\,du\\
&=
e^{-2h}\int_0^h ue^{2u}du.
\end{aligned}
$$

分部积分：

$$
\int ue^{2u}du
=
\frac u2e^{2u}-\frac14e^{2u}.
$$

因此：

$$
\begin{aligned}
J
&=
e^{-2h}
\left[
\left(\frac h2-\frac14\right)e^{2h}
+\frac14
\right]\\
&=
\frac h2-\frac14+\frac14e^{-2h}.
\end{aligned}
$$

整理：

$$
\boxed{
J
=
\frac{2h-1+e^{-2h}}4
}
\tag{8.11}
$$

代入第二项：

$$
\begin{aligned}
I_1
&=
2\alpha_t
\frac{D_s-D_r}{qh}
\frac{2h-1+e^{-2h}}4\\
&=
\alpha_t
\frac{2h-1+e^{-2h}}{2qh}
(D_s-D_r).
\end{aligned}
$$

因此：

$$
\boxed{
I_1
=
\alpha_t
\frac{2h-1+e^{-2h}}{2h}
\frac{D_s-D_r}{q}
}
\tag{8.12}
$$

##### 8.5 得到精确保留指数积分矩的二阶更新

将：

$$
I_0+I_1
$$

以及随机项代入：

$$
\boxed{
\begin{aligned}
x_t
={}&
\frac{\sigma_t}{\sigma_s}e^{-h}x_s\\
&+
\alpha_t(1-e^{-2h})D_s\\
&+
\alpha_t
\frac{2h-1+e^{-2h}}{2h}
\frac{D_s-D_r}{q}\\
&+
\sigma_t\sqrt{1-e^{-2h}}z.
\end{aligned}
}
\tag{8.13}
$$

其中：

$$
z\sim\mathcal N(0,I).
$$

这是一种基于历史线性插值、精确保留指数积分矩的二阶更新。

但**论文最终给出的式（20）使用了简化的二阶系数**。

##### 8.6 原论文为什么进一步简化？

考虑修正系数：

$$
A(h)
=
\frac{2h-1+e^{-2h}}{2h}.
$$

对：

$$
e^{-2h}
$$

进行 Taylor 展开：

$$
e^{-2h}
=
1-2h+2h^2-\frac43h^3+O(h^4).
$$

因此：

$$
\begin{aligned}
A(h)
&=
\frac{
2h-1+1-2h+2h^2-\frac43h^3+O(h^4)
}{2h}\\
&=
h-\frac23h^2+O(h^3).
\end{aligned}
\tag{8.14}
$$

另一方面：

$$
B(h)=\frac{1-e^{-2h}}2.
$$

展开：

$$
\begin{aligned}
B(h)
&=
\frac{
2h-2h^2+\frac43h^3+O(h^4)
}{2}\\
&=
h-h^2+\frac23h^3+O(h^4).
\end{aligned}
\tag{8.15}
$$

因此：

$$
\boxed{
A(h)-B(h)=O(h^2)
}
\tag{8.16}
$$

在相邻步长比例有界、历史预测差为 $O(h)$ 的光滑情形下，使用 $B(h)$ 替代 $A(h)$ 只引入更高阶的局部差异。

于是论文采用：

$$
\boxed{
A(h)\approx\frac{1-e^{-2h}}2
}
\tag{8.17}
$$

得到更简洁的更新。

##### 8.7 论文式（20）：SDE-DPM-Solver++(2M)

最终：

$$
\boxed{
\begin{aligned}
x_t
={}&
\frac{\sigma_t}{\sigma_s}e^{-h}x_s\\
&+
\alpha_t(1-e^{-2h})D_s\\
&+
\frac{\alpha_t}{2}(1-e^{-2h})
\frac{D_s-D_r}{q}\\
&+
\sigma_t\sqrt{1-e^{-2h}}z.
\end{aligned}
}
\tag{8.18}
$$

也可以定义：

$$
\boxed{
\bar D_s
=
D_s+\frac1{2q}(D_s-D_r)
}
\tag{8.19}
$$

则更新简化为：

$$
\boxed{
x_t
=
\frac{\sigma_t}{\sigma_s}e^{-h}x_s
+
\alpha_t(1-e^{-2h})\bar D_s
+
\sigma_t\sqrt{1-e^{-2h}}z
}
\tag{8.20}
$$

这就是 **SDE-DPM-Solver++(2M)** 的核心公式。[arXiv](https://arxiv.org/pdf/2211.01095)

它可以分解成三个部分：

$$
\boxed{
\underbrace{
\frac{\sigma_t}{\sigma_s}e^{-h}x_s
}_{\text{精确线性传播}}
+
\underbrace{
\alpha_t(1-e^{-2h})\bar D_s
}_{\text{二阶网络预测修正}}
+
\underbrace{
\sigma_t\sqrt{1-e^{-2h}}z
}_{\text{解析随机积分}}
}
\tag{8.21}
$$

这三个部分分别对应算法的三个关键设计。

---

#### 九、为什么 2M 每步只需要一次网络推理？

假设反向采样节点：

$$
t_0>t_1>\cdots>t_N.
$$

对应状态：

$$
x_0,x_1,\ldots,x_N.
$$

这里 $x_i$ 表示第 $i$ 个采样节点的状态，不要与真实数据 $x_0$ 混淆。

##### 9.1 第一轮

从初始高斯噪声：

$$
x_0^{\mathrm{sample}}\sim\mathcal N(0,I)
$$

开始。

计算：

$$
D_0=D_\theta(x_0^{\mathrm{sample}},t_0).
$$

由于没有历史预测：

$$
D_{-1}
$$

所以第一步使用 SDE-DPM-Solver++1：

$$
\boxed{
x_1
=
\frac{\sigma_1}{\sigma_0}e^{-h_1}x_0^{\mathrm{sample}}
+
\alpha_1(1-e^{-2h_1})D_0
+
\sigma_1\sqrt{1-e^{-2h_1}}z_0
}
\tag{9.1}
$$

##### 9.2 第二轮

计算：

$$
D_1=D_\theta(x_1,t_1).
$$

现在已经有：

$$
D_0,\quad D_1.
$$

于是可以使用二阶多步更新：

$$
\bar D_1
=
D_1+
\frac1{2q_1}(D_1-D_0).
$$

得到：

$$
x_2
=
\frac{\sigma_2}{\sigma_1}e^{-h_2}x_1
+
\alpha_2(1-e^{-2h_2})\bar D_1
+
\sigma_2\sqrt{1-e^{-2h_2}}z_1.
$$

##### 9.3 后续迭代

每一轮只需要：

1. 当前状态 $x_i$。
2. 当前网络预测 $D_i$。
3. 上一轮缓存的预测 $D_{i-1}$。
4. 新采样的高斯噪声 $z_i$。

所以：

$$
\boxed{
\text{每个新采样节点通常只需要一次网络推理}
}
$$

与确定性 DPM-Solver++(2M) 一样，2M 的效率主要来自复用历史网络输出。

但随机版本还需要每步采样一个新的：

$$
z_i\sim\mathcal N(0,I).
$$

高斯随机数生成通常远比大型神经网络推理便宜，因此不会显著改变 NFE。

---

#### 十、随机 2M 的“二阶”应该如何理解？

这里有一个比确定性 ODE 更容易混淆的问题。

对于 ODE，我们通常使用：

$$
\text{局部截断误差}
$$

和：

$$
\text{全局误差}
$$

来描述数值方法的阶数。

但对于 SDE，至少存在两类不同的误差概念。

##### 10.1 强收敛（Strong Convergence）

考虑同一个布朗运动路径下：

$$
X_T
$$

的精确解与：

$$
\hat X_T
$$

的数值解。

如果：

$$
\boxed{
\left(
\mathbb E\|X_T-\hat X_T\|^2
\right)^{1/2}
=
O(h^p)
}
\tag{10.1}
$$

则称数值方法具有均方意义下的强收敛阶 $p$。

它关注：

> 在相同随机驱动下，数值轨迹与真实随机轨迹有多接近？

##### 10.2 弱收敛（Weak Convergence）

对于足够光滑的测试函数：

$$
\varphi,
$$

如果：

$$
\boxed{
\left|
\mathbb E[\varphi(X_T)]
-
\mathbb E[\varphi(\hat X_T)]
\right|
=
O(h^p)
}
\tag{10.2}
$$

则称具有弱收敛阶 $p$。

它关注：

> 数值方法生成的概率分布是否接近真实 SDE 的概率分布？

##### 10.3 SDE-DPM-Solver++(2M) 的二阶不应简单理解为强二阶

论文称其为二阶求解器，主要是因为它对数据预测积分采用了一阶时间展开，并保留了相应的二阶漂移近似。

但是，**不能据此直接断言它具有标准意义下的强二阶收敛阶**。

原因在于：

$$
D_\theta(x_\lambda,\lambda)
$$

是沿随机轨迹变化的。

即使神经网络对 $x$ 和 $\lambda$ 足够光滑，由于：

$$
x_\lambda
$$

包含布朗运动，沿路径的变化也不是普通确定性 ODE 中那样光滑。

因此，严格的随机数值分析还需要考虑：

- Itô–Taylor 展开；
- 随机积分之间的相关性；
- 随机路径的正则性；
- 强误差与弱误差；
- 历史预测与当前随机增量的依赖关系。

论文在这里给出的是基于精确 SDE 积分表达式构造的高阶漂移近似，而不是对所有强、弱收敛阶都给出相同的保证。

这也是为什么不能将确定性 DPM-Solver 的：

$$
O(h_{\max}^2)
$$

收敛结论未经证明直接搬到 SDE-DPM-Solver++(2M) 上。

---

#### 十一、SDE-DPM-Solver++ 与 DDPM、DDIM、ODE-DPM-Solver++ 的区别

| 方法 | 求解对象 | 是否随机 | 主要数值处理 | 每步典型 NFE |
|---|---|---|---|---:|
| DDPM | 反向扩散马尔可夫链 | 是 | 离散高斯反向转移 | 1 |
| DDIM（$\eta=0$） | 确定性隐式采样过程 | 否 | 一阶确定性更新 | 1 |
| 随机 DDIM | 随机隐式采样过程 | 是 | 可调随机方差 | 1 |
| DPM-Solver++(2M) | PF ODE | 否 | 二阶数据预测指数多步 | 1 |
| SDE-DPM-Solver++1 | 反向 SDE | 是 | 一阶网络近似、解析随机方差 | 1 |
| SDE-DPM-Solver++(2M) | 反向 SDE | 是 | 二阶网络近似、解析随机方差 | 1 |

这里有三个需要特别注意的结论。

**第一，SDE-DPM-Solver++ 不是 DDPM 的简单改写。**

虽然二者都包含随机采样，但 DDPM 的标准反向更新来自离散扩散过程的反向高斯转移建模。

SDE-DPM-Solver++ 则从连续时间反向 SDE 出发，构造指数积分离散化。

**第二，SDE-DPM-Solver++1 与特定随机 DDIM 等价。**

这意味着它的一阶形式并不是完全独立于已有随机采样方法的更新。

其主要扩展价值在于进一步构造二阶多步方法。

**第三，SDE-DPM-Solver++(2M) 与确定性 DPM-Solver++(2M) 的二阶修正思想相似，但更新公式不同。**

确定性版本：

$$
\boxed{
x_t
=
\frac{\sigma_t}{\sigma_s}x_s
+
\alpha_t(1-e^{-h})\bar D_s
}
\tag{11.1}
$$

随机版本：

$$
\boxed{
x_t
=
\frac{\sigma_t}{\sigma_s}e^{-h}x_s
+
\alpha_t(1-e^{-2h})\bar D_s
+
\sigma_t\sqrt{1-e^{-2h}}z
}
\tag{11.2}
$$

不能通过简单地令：

$$
z=0
$$

就把随机版本变成 PF ODE 的求解器。

因为即使：

$$
z=0,
$$

二者的确定性漂移系数仍然不同。

---

#### 十二、为什么 SDE 采样可能更适合探索？

结合你之前研究的扩散策略强化学习，可以进一步理解随机采样的作用。

##### 12.1 ODE 采样

对于固定初始噪声：

$$
x_T,
$$

确定性 ODE 的输出为：

$$
\boxed{
x_0=\Phi_\theta(x_T,c)
}
\tag{12.1}
$$

在固定模型、条件和求解器的情况下，输出是确定的。

多模态主要来自不同的初始噪声：

$$
x_T^{(1)},\ldots,x_T^{(K)}.
$$

##### 12.2 SDE 采样

随机采样过程中：

$$
x_{i+1}
=
\mu_\theta(x_i,c,t_i)
+
s_i z_i.
$$

因此即使固定：

$$
x_T,
$$

只要中间的：

$$
z_i
$$

不同，也可能生成不同的轨迹。

其随机性来自：

$$
\boxed{
\text{初始噪声}
+
\text{反向过程中的随机增量}
}
\tag{12.2}
$$

但要注意：

**随机性来源更多，不意味着最终生成分布一定具有更高的有效多模态性。**

如果模型和 Score 足够准确，ODE 与 SDE 理论上可以具有相同的最终边缘分布。

它们的主要区别在于路径随机性和数值离散化行为。

##### 12.3 对轨迹规划的意义

对于自动驾驶轨迹规划，设：

$$
Y\in\mathbb R^{T\times d}
$$

为未来轨迹，条件：

$$
c
$$

为场景编码。

如果使用 VP 型扩散模型，可以采用 SDE-DPM-Solver++ 生成：

$$
\boxed{
Y^{(1)},Y^{(2)},\ldots,Y^{(K)}
}
$$

然后根据：

- 碰撞风险；
- 交通规则；
- 舒适性；
- 导航一致性；
- 行驶效率；

计算奖励或评价指标。

随机 SDE 求解器提供了另一种探索方式。

不过在规划任务中，随机扰动也可能增加不可行轨迹的比例，因此需要通过闭环实验判断其收益。

---

#### 十三、与 Diffusion GRPO 的联系：每一步都有显式高斯转移

这是 SDE-DPM-Solver++ 对你研究的 RL 框架可能最有价值的地方。

##### 13.1 一阶采样器的转移概率

根据式（6.4）：

$$
x_t\mid x_s,c
\sim
\mathcal N
\left(
\mu_\theta(x_s,c),s_{s,t}^2I
\right),
$$

其中：

$$
\boxed{
s_{s,t}^2
=
\sigma_t^2(1-e^{-2h})
}
\tag{13.1}
$$

因此每一步的条件对数概率可以直接计算：

$$
\boxed{
\begin{aligned}
\log p_\theta(x_t\mid x_s,c)
={}&
-\frac{\|x_t-\mu_\theta(x_s,c)\|^2}
{2s_{s,t}^2}\\
&-
\frac D2\log(2\pi s_{s,t}^2).
\end{aligned}
}
\tag{13.2}
$$

其中 $D$ 是采样状态的总维度。

例如轨迹：

$$
x_t\in\mathbb R^{T\times2},
$$

则：

$$
D=2T.
$$

##### 13.2 二阶多步采样器的转移概率

对于 SDE-DPM-Solver++(2M)：

$$
\bar D_s
=
D_s+\frac1{2q}(D_s-D_r).
$$

更新：

$$
x_t
=
\mu_\theta(x_s,x_r,c)
+
s_{s,t}z.
$$

这里为了避免符号歧义，将条件均值明确写为：

$$
\boxed{
\mu_\theta(x_s,x_r,c)
=
\frac{\sigma_t}{\sigma_s}e^{-h}x_s
+
\alpha_t(1-e^{-2h})\bar D_s
}
\tag{13.3}
$$

因此：

$$
\boxed{
p_\theta(x_t\mid x_s,x_r,c)
=
\mathcal N
\left(
\mu_\theta(x_s,x_r,c),
s_{s,t}^2I
\right)
}
\tag{13.4}
$$

注意：

**二阶多步采样器的单步转移不仅依赖当前状态，还依赖历史状态或历史网络预测。**

因此：

$$
p_\theta(x_t\mid x_s,c)
$$

一般不足以完整描述 2M 的一步转移。

更准确的是：

$$
\boxed{
p_\theta(x_t\mid x_s,x_r,c)
}
\tag{13.5}
$$

如果希望将它作为强化学习策略，可以将历史状态纳入扩展状态：

$$
\boxed{
S_i=(x_i,x_{i-1},t_i)
}
\tag{13.6}
$$

这样才能在扩展状态空间中恢复马尔可夫形式。

##### 13.3 整条采样链的概率

假设采样过程为：

$$
x_N\rightarrow x_{N-1}\rightarrow\cdots\rightarrow x_0.
$$

对于一阶马尔可夫采样器：

$$
\boxed{
p_\theta(x_{0:N}\mid c)
=
p(x_N)
\prod_{i=N}^{1}
p_\theta(x_{i-1}\mid x_i,c)
}
\tag{13.7}
$$

对于二阶多步采样器，必须保留历史依赖。

如果将采样方向重新编号为：

$$
y_0\rightarrow y_1\rightarrow\cdots\rightarrow y_N,
$$

则：

$$
\boxed{
p_\theta(y_{0:N}\mid c)
=
p(y_0)
p_\theta(y_1\mid y_0,c)
\prod_{i=1}^{N-1}
p_\theta(y_{i+1}\mid y_i,y_{i-1},c)
}
\tag{13.8}
$$

这意味着：

$$
\boxed{
\log p_\theta(y_{0:N}\mid c)
=
\log p(y_0)
+
\sum_i\log p_\theta(\text{对应转移})
}
\tag{13.9}
$$

因此，如果将完整去噪链作为强化学习中的动作序列，就可以计算路径概率以及新旧策略之间的路径概率比。

##### 13.4 这是否等于最终轨迹的概率密度？

不等于。

式（13.9）给出的是：

$$
\boxed{
\text{完整采样路径的联合概率密度}
}
$$

而最终轨迹：

$$
y_N
$$

的边缘概率密度为：

$$
\boxed{
p_\theta(y_N\mid c)
=
\int
p_\theta(y_{0:N}\mid c)
\,dy_0\cdots dy_{N-1}
}
\tag{13.10}
$$

通常不能直接把路径上各步的条件密度相乘，就当作最终轨迹的边缘密度。

对于 Diffusion GRPO、PPO 一类算法，可以选择在完整采样链的条件转移上构造策略梯度目标，但必须明确优化的是哪一种路径策略分布，以及最终奖励如何分配给各步。

此外，由于 2M 依赖历史预测，计算新旧策略的概率比时，需要在**同一条已采样路径及其历史状态**上重新计算对应条件均值，不能只替换当前一步的网络输出而忽略历史模型预测的变化。

---

#### 十四、为什么不能直接用于任意 Flow Matching？

这里要区分两种情况。

##### 14.1 VP 扩散模型

如果模型满足：

$$
x_t=\alpha_t x_0+\sigma_t\epsilon,
$$

并且：

$$
\alpha_t^2+\sigma_t^2=1,
$$

那么论文 Section 5 的 SDE 推导可以直接应用。

##### 14.2 一般 Flow Matching

对于：

$$
\frac{dx_t}{dt}
=
v_\theta(x_t,t,c),
$$

即使使用：

$$
x_t=(1-t)z+ty,
$$

也不能直接把 SDE-DPM-Solver++ 的公式照搬过来。

因为：

$$
\boxed{
\text{给定 ODE 的边缘概率路径}
}
$$

并不能唯一确定：

$$
\boxed{
\text{反向 SDE 的扩散系数}
}
$$

若希望构造等价 SDE，需要明确：

1. 当前 ODE 对应的边缘分布 $p_t$。
2. 希望使用的扩散系数。
3. 相应的 Score 或能够恢复 Score 的模型参数化。
4. 由 Fokker–Planck 方程确定的修正漂移。

在适当条件下，对于：

$$
\frac{dx_t}{dt}=v_t(x_t)
$$

对应的边缘分布，可以构造具有相同边缘分布的 SDE：

$$
\boxed{
dx_t
=
\left[
v_t(x_t)+\frac12a(t)\nabla_x\log p_t(x_t)
\right]dt
+
\sqrt{a(t)}\,dW_t
}
\tag{14.1}
$$

这里采用的是与 ODE 相同的正向时间方向，并假设扩散矩阵为：

$$
a(t)I.
$$

但这只是等价 SDE 的一种构造。

它不意味着 SDE-DPM-Solver++ 的 VP 专用系数：

$$
\frac{\sigma_t}{\sigma_s}e^{-h},
\qquad
\alpha_t(1-e^{-2h}),
\qquad
\sigma_t\sqrt{1-e^{-2h}}
$$

可以直接沿用。

如果你希望将它应用到 Conditional Flow Matching 轨迹规划模型，必须重新推导相应的积分表达式。

---

#### 十五、总结：SDE-DPM-Solver++ 最重要的五个结论

**1. 它求解的是反向扩散 SDE，而不是 PF ODE。**

反向 SDE：

$$
dx_t=
\left[
f(t)x_t+
\frac{g^2(t)}{\sigma_t}\epsilon_\theta
\right]dt
+
g(t)d\bar W_t.
$$

**2. 它使用 logSNR 变量变换，将 VP 反向 SDE 写成具有已知线性结构的形式。**

$$
\boxed{
dx_\lambda
=
\left[
-(1+\alpha_\lambda^2)x_\lambda
+
2\alpha_\lambda D_\theta
\right]d\lambda
+
\sqrt2\sigma_\lambda dW_\lambda
}
$$

**3. 它能够解析处理线性传播和随机积分。**

最终精确积分表达式：

$$
\boxed{
\begin{aligned}
x_t
={}&
\frac{\sigma_t}{\sigma_s}e^{-h}x_s\\
&+
2\alpha_t
\int_{\lambda_s}^{\lambda_t}
e^{-2(\lambda_t-\lambda)}
D_\theta(x_\lambda,\lambda)d\lambda\\
&+
\sqrt2\sigma_t
\int_{\lambda_s}^{\lambda_t}
e^{-(\lambda_t-\lambda)}dW_\lambda.
\end{aligned}
}
$$

**4. 一阶版本等价于采用特定噪声尺度的随机 DDIM。**

$$
\boxed{
x_t
=
\frac{\sigma_t}{\sigma_s}e^{-h}x_s
+
\alpha_t(1-e^{-2h})D_s
+
\sigma_t\sqrt{1-e^{-2h}}z
}
$$

**5. 二阶多步版本复用历史网络预测，通常每个新节点只需要一次网络推理。**

$$
\boxed{
\begin{aligned}
\bar D_s
&=
D_s+\frac1{2q}(D_s-D_r),\\
x_t
&=
\frac{\sigma_t}{\sigma_s}e^{-h}x_s\\
&\quad+
\alpha_t(1-e^{-2h})\bar D_s\\
&\quad+
\sigma_t\sqrt{1-e^{-2h}}z.
\end{aligned}
}
$$

对于你的扩散轨迹规划与强化学习研究，最值得关注的是最后一点：**SDE-DPM-Solver++(2M) 同时具备高效的多步采样结构和显式的逐步高斯转移密度，因此可以作为基于完整去噪路径构造策略梯度目标的一种候选采样器。**

不过，使用它时必须保留历史状态依赖，并区分采样路径概率与最终轨迹的边缘概率。

**参考论文：** [DPM-Solver++，Section 5、6.1 与 Appendix A](https://arxiv.org/pdf/2211.01095)。

---



## 19. DPM-Solver++ 的实验结论

DPM-Solver++ 论文主要围绕强引导采样进行实验。

研究对象包括：

- 像素空间扩散模型；
- 潜空间扩散模型；
- Classifier Guidance；
- Classifier-Free Guidance。

论文重点考察三项改进的作用。

### 19.1 数据预测参数化

实验比较：

$$
\epsilon_\theta
$$

与：

$$
D_\theta
$$

对应的高阶求解器。

结果表明，在论文测试的强引导设置下，数据预测形式能够改善少步数采样的稳定性与质量。

但这并不等于：

$$
\boxed{
\text{数据预测在任何求解器下都优于噪声预测}
}
$$

论文比较的是特定积分表达式与高阶数值方法的组合。

### 19.2 Singlestep 与 Multistep

实验比较：

$$
\text{DPM-Solver++(2S)}
$$

和：

$$
\text{DPM-Solver++(2M)}.
$$

结果显示，2M 在相同 NFE 预算下通常更有效。

主要原因是：

$$
\boxed{
\text{复用历史网络输出}
\rightarrow
\text{更多积分节点}
\rightarrow
\text{更小的单步长度}
}
$$

### 19.3 Thresholding

对于具有明确数据边界的像素空间模型，适当的 Thresholding 可以改善强引导造成的数据范围失配问题。

但对于潜空间模型，Thresholding 的适用性需要单独判断。

### 19.4 最终实验结论

论文报告，DPM-Solver++ 能够在多种强引导图像生成任务中，以大约 15–20 次网络计算得到较好的生成结果。

其核心价值是：

$$
\boxed{
\text{无需重新训练模型，改善强引导下的少步数采样}
}
$$

论文还通过消融实验分别验证了数据预测参数化、多步更新和 Thresholding 的贡献。[arXiv](https://arxiv.org/pdf/2211.01095?utm_source=chatgpt.com)

---

# 第三部分：与其他扩散采样方法的联系

## 20. DPM-Solver、DPM-Solver++、DDIM、Heun 的关系

现在可以将这些方法放在统一框架下理解。

| 方法 | 主要积分对象 | 阶数 | 是否使用历史预测 | 每步典型 NFE |
|---|---|---:|---|---:|
| DDIM（$\eta=0$） | 噪声或数据预测 | 1 | 否 | 1 |
| DPM-Solver-1 | 噪声预测 | 1 | 否 | 1 |
| DPM-Solver-2 | 噪声预测 | 2 | 否 | 2 |
| DPM-Solver-3 | 噪声预测 | 3 | 否 | 3 |
| DPM-Solver++(2S) | 数据预测 | 2 | 否 | 2 |
| DPM-Solver++(2M) | 数据预测 | 2 | 是 | 通常 1 |
| Heun | 完整 ODE 速度场 | 2 | 否 | 通常 2 |

需要特别理解三个区别。

**第一，DDIM 是一阶指数积分方法。**

$$
\boxed{
\text{DDIM}
=
\text{DPM-Solver 的一阶特例}
}
$$

**第二，DPM-Solver 和 DPM-Solver++ 的高阶版本并不等价。**

虽然：

$$
D_\theta
=
\frac{x-\sigma\epsilon_\theta}{\alpha},
$$

但在有限步长下：

$$
\text{对 }\epsilon_\theta\text{ 做高阶近似}
$$

与：

$$
\text{对 }D_\theta\text{ 做高阶近似}
$$

会得到不同的数值方法。

**第三，2S 和 2M 的区别主要是如何获得高阶变化信息。**

2S：

$$
\boxed{
\text{当前网络预测}
+
\text{中间点网络预测}
}
$$

2M：

$$
\boxed{
\text{当前网络预测}
+
\text{历史网络预测}
}
$$

---

## 21. 与 EDM 的联系

上一章 EDM 推荐的 ODE 为：

$$
\boxed{
\frac{dx}{d\sigma}
=
\frac{x-D_\theta(x,\sigma)}{\sigma}
}
\tag{21.1}
$$

这里使用 EDM 的 VE 风格参数化：

$$
\alpha=1,
\qquad
\sigma_{\mathrm{noise}}=\sigma.
$$

注意，为避免与一般扩散调度混淆，这里的 $\sigma$ 表示 EDM 的噪声标准差。

### 21.1 DPM-Solver++ 能否应用于 EDM？

可以从数学形式上建立联系。

对于：

$$
\alpha=1,
$$

有：

$$
\lambda=\log\frac1{\sigma}=-\log\sigma.
$$

因此：

$$
\sigma=e^{-\lambda}.
$$

DPM-Solver++ 的精确积分形式：

$$
x_t
=
\frac{\sigma_t}{\sigma_s}x_s
+
\sigma_t
\int_{\lambda_s}^{\lambda_t}
e^\lambda D_\theta(x_\lambda,\lambda)d\lambda
$$

仍然可以用于对应的 VE 型概率路径。

其一阶更新为：

$$
\boxed{
x_t
=
\frac{\sigma_t}{\sigma_s}x_s
+
\left(
1-\frac{\sigma_t}{\sigma_s}
\right)D_\theta(x_s,\sigma_s)
}
\tag{21.2}
$$

因为：

$$
\alpha_t=1,
$$

并且：

$$
e^{-h}=\frac{\sigma_t}{\sigma_s}.
$$

整理：

$$
\boxed{
x_t
=
x_s+
\left(
1-\frac{\sigma_t}{\sigma_s}
\right)
(D_\theta-x_s)
}
\tag{21.3}
$$

而 EDM 的 Euler 更新：

$$
x_t
=
x_s+
(\sigma_t-\sigma_s)
\frac{x_s-D_\theta}{\sigma_s}
$$

可以整理为：

$$
\begin{aligned}
x_t
&=
x_s+
\left(
\frac{\sigma_t}{\sigma_s}-1
\right)(x_s-D_\theta)\\
&=
x_s+
\left(
1-\frac{\sigma_t}{\sigma_s}
\right)(D_\theta-x_s).
\end{aligned}
$$

因此：

$$
\boxed{
\text{EDM VE 参数化下的 Euler}
=
\text{对应数据预测形式的一阶 DPM-Solver++}
}
\tag{21.4}
$$

这是一个非常有价值的联系。

不过高阶方法并不因此自动等价：

$$
\boxed{
\text{EDM Heun}
\neq
\text{DPM-Solver++(2S/2M)}
}
$$

因为它们对网络输出随时间变化的近似方式不同。

---

## 22. 为什么 DPM-Solver 不一定适合标准 Flow Matching？

对于一般 Conditional Flow Matching：

$$
\boxed{
\frac{dx_t}{dt}
=
v_\theta(x_t,t,c)
}
\tag{22.1}
$$

例如采用：

$$
x_t=(1-t)z+ty,
$$

其中：

$$
z\sim\mathcal N(0,I),
\qquad
y\sim p_{\mathrm{data}}.
$$

训练目标：

$$
\boxed{
u_t=y-z
}
\tag{22.2}
$$

模型学习：

$$
v_\theta(x_t,t,c)\approx
\mathbb E[y-z\mid x_t,c].
$$

这种 ODE 并不天然写成 DPM-Solver 原论文使用的噪声预测半线性形式：

$$
f(t)x_t+b(t)\epsilon_\theta(x_t,t).
$$

因此不能直接将 DPM-Solver 的更新公式照搬到任意 Flow Matching 模型。

但如果能够建立：

$$
\boxed{
v_\theta(x,t)
=
a(t)x+b(t)F_\theta(x,t)
}
\tag{22.3}
$$

并且：

$$
a(t)
$$

对应的线性部分可以解析求解，那么就可以考虑使用指数积分方法。

### 22.1 对于线性插值 Flow Matching

设：

$$
x_t=(1-t)z+ty.
$$

可以定义：

$$
\alpha_t=t,
\qquad
\sigma_t=1-t.
$$

于是：

$$
x_t=\alpha_t y+\sigma_t z.
$$

定义：

$$
\lambda_t=\log\frac{t}{1-t}.
$$

从数学上看，这与 DPM-Solver 的高斯路径参数化具有联系。

但这里需要注意时间方向。

标准 Flow Matching 通常：

$$
t:0\rightarrow1
$$

表示从噪声到数据。

而 DPM-Solver 原论文：

$$
t:T\rightarrow0
$$

表示从噪声到数据。

因此若要复用公式，必须统一时间方向、噪声调度和网络预测定义。

此外，Flow Matching 网络通常直接预测：

$$
v_\theta,
$$

而不是：

$$
\epsilon_\theta
$$

或：

$$
D_\theta.
$$

需要先建立对应转换关系，才能判断是否能够使用 DPM-Solver 的指数积分公式。

---

# 第四部分：对自动驾驶轨迹规划的启发

## 23. DPM-Solver++(2M) 为什么值得用于轨迹规划？

对于轨迹规划：

$$
Y\in\mathbb R^{T\times d},
$$

其中 $T$ 是未来轨迹长度，$d$ 是单个轨迹点的维度。

条件：

$$
c
$$

包含感知、地图、导航和自车状态。

假设生成模型为：

$$
\boxed{
\frac{dx_t}{dt}
=
v_\theta(x_t,t,c)
}
$$

并且需要：

$$
N
$$

次积分更新。

如果使用 Heun：

$$
\mathrm{NFE}\approx2N.
$$

如果能够构造合适的二阶多步方法：

$$
\mathrm{NFE}\approx N.
$$

对于需要实时推理的自动驾驶规划器，减少网络调用次数可能直接降低推理延迟。

但具体收益还取决于网络前向开销、batch 大小和硬件执行效率。

## 24. 应该优先研究什么？

结合 DPM-Solver 的思想，我认为可以将研究问题分为三个层次。

### 第一层：保持模型不变，只研究采样器

比较：

$$
\text{Euler}
$$

$$
\text{Heun}
$$

$$
\text{二阶多步方法}
$$

在相同 NFE 下的表现。

评价指标不应只有轨迹 L2 误差，还应包含：

- 碰撞率；
- 轨迹可行性；
- 舒适性；
- 多模态覆盖；
- 推理延迟。

### 第二层：研究预测空间

比较：

$$
v_\theta
$$

与：

$$
D_\theta
$$

等不同参数化。

重点不是简单判断哪个预测目标更好，而是分析：

$$
\boxed{
\text{预测空间}
+
\text{数值积分方法}
}
$$

是否匹配。

这与 EDM 中网络预测空间和 Loss 空间可以分别设计的思想一致。

### 第三层：研究时间网格

可以比较：

$$
t\text{-uniform}
$$

$$
\lambda\text{-uniform}
$$

以及其他非均匀网格。

但对于一般 Flow Matching，不能直接假定：

$$
\lambda=\log\frac{\alpha}{\sigma}
$$

一定是最合适的积分时间变量。

是否采用 logSNR，应根据具体概率路径和 ODE 结构决定。

---

# 第五部分：全文核心公式总结

## 25. DPM-Solver

高斯边缘分布：

$$
\boxed{
x_t=\alpha_t x_0+\sigma_t\epsilon
}
$$

半对数信噪比：

$$
\boxed{
\lambda_t=\log\frac{\alpha_t}{\sigma_t}
}
$$

PF ODE：

$$
\boxed{
\frac{dx_t}{dt}
=
f(t)x_t+
\frac{g^2(t)}{2\sigma_t}
\epsilon_\theta(x_t,t)
}
$$

精确积分：

$$
\boxed{
x_t
=
\frac{\alpha_t}{\alpha_s}x_s
-
\alpha_t
\int_{\lambda_s}^{\lambda_t}
e^{-\lambda}\epsilon_\theta(x_\lambda,\lambda)d\lambda
}
$$

一阶更新：

$$
\boxed{
x_t
=
\frac{\alpha_t}{\alpha_s}x_s
-
\sigma_t(e^h-1)\epsilon_\theta(x_s,s)
}
$$

二阶指数中点更新：

$$
\boxed{
\begin{aligned}
u
&=
\frac{\alpha_m}{\alpha_s}x_s
-
\sigma_m(e^{h/2}-1)\epsilon_\theta(x_s,s),\\
x_t
&=
\frac{\alpha_t}{\alpha_s}x_s
-
\sigma_t(e^h-1)\epsilon_\theta(u,m).
\end{aligned}
}
$$

收敛阶：

$$
\boxed{
\text{DPM-Solver-}k:
\quad O(h_{\max}^k),\quad k=1,2,3
}
$$

## 26. DPM-Solver++

数据预测：

$$
\boxed{
D_\theta(x_t,t)
=
\frac{x_t-\sigma_t\epsilon_\theta(x_t,t)}
{\alpha_t}
}
$$

精确积分：

$$
\boxed{
x_t
=
\frac{\sigma_t}{\sigma_s}x_s
+
\sigma_t
\int_{\lambda_s}^{\lambda_t}
e^\lambda D_\theta(x_\lambda,\lambda)d\lambda
}
$$

一阶更新：

$$
\boxed{
x_t
=
\frac{\sigma_t}{\sigma_s}x_s
+
\alpha_t(1-e^{-h})D_\theta(x_s,s)
}
$$

二阶单步：

$$
\boxed{
\begin{aligned}
u
&=
\frac{\sigma_m}{\sigma_s}x_s
+
\alpha_m(1-e^{-rh})D_s,\\
\bar D
&=
\left(1-\frac1{2r}\right)D_s
+
\frac1{2r}D_m,\\
x_t
&=
\frac{\sigma_t}{\sigma_s}x_s
+
\alpha_t(1-e^{-h})\bar D.
\end{aligned}
}
$$

二阶多步：

$$
\boxed{
\begin{aligned}
r_i&=\frac{h_{i-1}}{h_i},\\
\bar D_i
&=
\left(1+\frac1{2r_i}\right)D_{i-1}
-
\frac1{2r_i}D_{i-2},\\
x_i
&=
\frac{\sigma_i}{\sigma_{i-1}}x_{i-1}
+
\alpha_i(1-e^{-h_i})\bar D_i.
\end{aligned}
}
$$

---

# 六、最终总结

DPM-Solver 系列的核心不是单纯提高 ODE 求解器的阶数，而是**利用扩散模型本身的数学结构，重新设计数值积分方法**。

原始 DPM-Solver 发现：

$$
\boxed{
\text{扩散 PF ODE}
=
\text{已知线性项}
+
\text{神经网络非线性项}
}
$$

因此可以先解析处理线性项，再通过 logSNR 变量替换，将剩余问题变成指数加权积分：

$$
\boxed{
\int e^{-\lambda}\epsilon_\theta\,d\lambda
}
$$

随后使用一阶、二阶和三阶方法近似网络预测随时间的变化。

DPM-Solver++ 则进一步发现，在强引导采样中，使用数据预测积分：

$$
\boxed{
\int e^\lambda D_\theta\,d\lambda
}
$$

并结合 Thresholding 和 Multistep，可以改善少步数生成的稳定性与质量。

其中，DPM-Solver++(2M) 最值得关注的特点是：

$$
\boxed{
\text{通过复用历史网络预测，以约一次新增 NFE 完成一次二阶多步更新}
}
$$

这也是它在计算预算受限的生成任务中具有实际价值的原因。

**两篇论文最重要的统一结论是：数值求解器的设计不能脱离生成模型的概率路径、预测参数化和时间变量。** 同样是二阶方法，对完整速度场积分、对噪声预测积分、对数据预测积分，得到的数值算法及误差表现都可能不同。

### 原论文

- [DPM-Solver（NeurIPS 2022）](https://arxiv.org/abs/2206.00927)
- [DPM-Solver++](https://arxiv.org/abs/2211.01095)
- [官方代码：LuChengTHU/dpm-solver](https://github.com/LuChengTHU/dpm-solver)